# TANAHI-main — Hệ thống nhận diện sản phẩm URA Hackathon 2026

Kết hợp **have_train** (rules + ML trên OCR) với **pipeline layout** (khung đỏ/xanh + bố cục):

- **OCR:** Paddle **chỉ det** (PP-OCRv4 bound box) + **VietOCR** đọc chữ tiếng Việt/Anh trên crop — **không dùng latin rec Paddle**.
- **Dự đoán chính:** `predict_labels` (rules → alias → BrandClassifier → ProductClassifier → KNN) giống have_train.
- **Pipeline layout** (`extract_v5` + khung đỏ): chỉ **bù** khi prominence cao và ML trống hoặc brand/product hiếm/chưa có trong train — **không ghi đè** ML đã có support.

| # | Chức năng |
|---|----------|
| 1 | Cài đặt thư viện |
| 2 | Cấu hình + ngưỡng |
| 3 | Rules/regex + chuẩn hoá (have_train) |
| 4 | Lõi layout: khung đỏ + extract_v5 |
| 5 | Huấn luyện ML + học động |
| 6 | `predict_labels` + `predict_image` + eval |
| 7 | Paddle det + VietOCR (không latin) |
| 8 | Chạy inference + checkpoint |
| 9 | Hậu kỳ nhẹ product_name |
| 10 | Xuất submission.csv |


In [ ]:
# ============================================================
# CELL 1 — CÀI ĐẶT THƯ VIỆN
# Giữ numpy mặc định Kaggle (thường 2.x). Không hạ numpy — chỉ cài opencv
# headless CUỐI cùng để khớp ABI (sửa lỗi import cv2 ở cell 4).
# KHÔNG import pandas/numpy/cv2 ở cell này.
# ============================================================
print("CELL 1 | Đang cài đặt thư viện, vui lòng đợi 2-3 phút...")

import glob, os, shutil, site, sys, subprocess

def _pip(*pkgs, force=False):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
    if force:
        cmd.append("--force-reinstall")
    cmd.extend(pkgs)
    subprocess.run(cmd, check=False)

def _scrub_pillow_from_site():
    """pip uninstall không gỡ .so cũ — xóa PIL thủ công tránh lệch _imaging vs __version__."""
    roots = set(site.getsitepackages())
    try:
        roots.add(site.getusersitepackages())
    except Exception:
        pass
    for root in roots:
        pil_dir = os.path.join(root, "PIL")
        if os.path.isdir(pil_dir):
            shutil.rmtree(pil_dir, ignore_errors=True)
        for pat in ("pillow-*.dist-info", "Pillow-*.dist-info"):
            for p in glob.glob(os.path.join(root, pat)):
                shutil.rmtree(p, ignore_errors=True)

_pip(
    "paddlepaddle", "paddleocr==2.7.3", "vietocr",
    "scikit-learn", "joblib", "tqdm", "pandas", "requests",
)

# Gỡ opencv cũ (Kaggle hay lệch ABI) -> cài headless khớp numpy hiện tại
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "opencv-python", "opencv-contrib-python", "opencv-python-headless"],
    check=False,
)
_pip("opencv-python-headless>=4.10.0", force=True)

# Pillow: 11.3.0 (PaddleOCR 2.7.3 OK; tránh Pillow>=12). Cài sạch sau vietocr.
PILLOW_VER = "11.3.0"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pillow"], check=False)
_scrub_pillow_from_site()
_pip(f"pillow=={PILLOW_VER}", force=True)

print(f"Đã cài đặt xong (opencv-headless + pillow=={PILLOW_VER}, PIL scrub sạch).")
print("✓ Đã chạy xong cell 1")

In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH & DÒ ĐƯỜNG DẪN DỮ LIỆU
# Tự động tìm test/train/sample CSV + thư mục ảnh (test & train),
# chuẩn hoá cột id/schema + ngưỡng layout/ học động.
# Chạy cell 1 trước. Nếu lỗi pandas/numpy -> Restart session -> Run All.
# ============================================================
import os, re, csv, time, warnings, unicodedata
import numpy as np          # import numpy TRƯỚC pandas
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image, ImageEnhance, ImageFilter
from collections import Counter
warnings.filterwarnings('ignore')

# ---- Cấu hình chính ----
RUNNING_ON_KAGGLE      = True    # True = Kaggle, False = chạy máy cá nhân
OCR_BOX_MIN_AREA_RATIO     = 0.003   # lọc box det < 0.3% diện tích ảnh
PIPELINE_LAYOUT_MIN_PROMINENCE = 30.0  # layout chỉ bù ML khi prominence >= ngưỡng này
PIPELINE_LAYOUT_FALLBACK_BRAND = 22.0  # ngưỡng fallback brand trong extract_v5
DYNAMIC_PROMINENCE_THRESHOLD = 28.0  # học động: đăng ký brand/product mới khi prominence > ngưỡng
SKIP_TRAIN_DISCOVERY       = False   # True = bỏ qua bước học brand mới từ ảnh train
TRAIN_DISCOVERY_SAMPLE     = 100     # số ảnh train tối đa dùng để khám phá brand mới

CPU_THREADS = max(1, os.cpu_count() or 2)
NUM_WORKERS = max(1, min(CPU_THREADS, 4))
for _env in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'):
    os.environ.setdefault(_env, str(CPU_THREADS))


def _find_first(root, names):
    """Tìm đệ quy file theo danh sách tên ưu tiên (vd private_test.csv trước test.csv)."""
    if root is None or not Path(root).exists():
        return None
    root = Path(root)
    for nm in names:
        hits = sorted(root.rglob(nm))
        if hits:
            return hits[0]
    return None


def _find_images_dir(root, prefer=('test',)):
    """Trả về thư mục chứa nhiều ảnh nhất (đệ quy). Ưu tiên thư mục có từ khoá
    trong `prefer` để tách biệt ảnh test và ảnh train."""
    if root is None or not Path(root).exists():
        return None
    root = Path(root)
    items = []
    cands = [root] + [p for p in root.rglob('*') if p.is_dir()]
    for d in cands:
        try:
            n = sum(1 for p in d.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
        except Exception:
            n = 0
        if n > 0:
            items.append((d, n))
    if not items:
        return None
    pref = [t for t in items if any(k in str(t[0]).lower() for k in prefer)]
    pool = pref if pref else items
    return max(pool, key=lambda t: t[1])[0]


if RUNNING_ON_KAGGLE:
    _roots = [
        Path('/kaggle/input/competitions/the-2nd-ura-hackathon'),
        Path('/kaggle/input/the-2nd-ura-hackathon'),
    ]
    COMP_ROOT = next((p for p in _roots if p.exists()), _roots[0])
    WORK_DIR  = Path('/kaggle/working')
else:
    COMP_ROOT = Path(r'C:\Users\LENOVO\Downloads\Urahackathon2026\the-2nd-ura-hackathon')
    WORK_DIR  = Path('.')

TEST_CSV   = _find_first(COMP_ROOT, ['private_test.csv', 'test_private.csv',
                                     'public_test.csv', 'test.csv'])
SAMPLE_CSV = _find_first(COMP_ROOT, ['sample_submission_private.csv', 'sample_submission.csv',
                                     'sample_submission_public.csv'])
TRAIN_CSV  = _find_first(COMP_ROOT, ['train_labels.csv', 'train.csv', 'train_label.csv'])

# Ảnh test: ưu tiên cùng nhánh với file test (tránh nhầm ảnh train)
_test_base = TEST_CSV.parent if TEST_CSV is not None else COMP_ROOT
IMAGES_DIR = (_find_images_dir(_test_base, prefer=('test',))
              or _find_images_dir(COMP_ROOT, prefer=('test',)) or COMP_ROOT)
# Ảnh train: phục vụ học động (discovery)
TRAIN_IMAGES_DIR = ((_find_images_dir(TRAIN_CSV.parent, prefer=('train',)) if TRAIN_CSV is not None else None)
                    or _find_images_dir(COMP_ROOT, prefer=('train',)))

OUTPUT_CSV     = WORK_DIR / 'submission.csv'
CHECKPOINT_CSV = WORK_DIR / 'checkpoint.csv'

assert TEST_CSV is not None and TEST_CSV.exists(), \
    'Không tìm thấy file test csv dưới %s. Hãy kiểm tra đã Add dataset chưa.' % COMP_ROOT

test_df = pd.read_csv(TEST_CSV)
train_labels_df = pd.read_csv(TRAIN_CSV, keep_default_na=False) if (TRAIN_CSV and TRAIN_CSV.exists()) else None


def _detect_col(df, *cands):
    if df is None:
        return None
    low = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None

# Chuẩn hoá cột id của test về 'image_id'
_TEST_ID = _detect_col(test_df, 'image_id', 'id', 'img_id', 'image', 'filename', 'file_name')
if _TEST_ID and _TEST_ID != 'image_id':
    test_df = test_df.rename(columns={_TEST_ID: 'image_id'})

BRAND_COL     = _detect_col(train_labels_df, 'brand_name', 'brand', 'thuong_hieu')
PRODUCT_COL   = _detect_col(train_labels_df, 'product_name', 'product', 'san_pham')
OCR_COL       = _detect_col(train_labels_df, 'ocr_text', 'ocr', 'text')
HAS_BRAND_COL = BRAND_COL is not None

# Sample submission quyết định thứ tự / tên cột đầu ra
if SAMPLE_CSV is not None and SAMPLE_CSV.exists():
    _sample_head = pd.read_csv(SAMPLE_CSV, keep_default_na=False, nrows=1)
    SUBMISSION_COLS = list(_sample_head.columns)
else:
    SUBMISSION_COLS = ['image_id', 'ocr_text', 'brand_name', 'product_name']

# Lập chỉ mục đường dẫn ảnh test
IMAGE_PATH_INDEX = {}
if IMAGES_DIR and Path(IMAGES_DIR).is_dir():
    for p in Path(IMAGES_DIR).glob('*'):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            IMAGE_PATH_INDEX[p.stem] = str(p)
            IMAGE_PATH_INDEX[p.name] = str(p)

print('Dò đường dẫn dữ liệu:')
print(f'  COMP_ROOT        : {COMP_ROOT} (tồn tại={COMP_ROOT.exists()})')
print(f'  TEST_CSV         : {TEST_CSV}')
print(f'  TRAIN_CSV        : {TRAIN_CSV}')
print(f'  SAMPLE_CSV       : {SAMPLE_CSV}')
print(f'  IMAGES_DIR (test): {IMAGES_DIR}')
print(f'  TRAIN_IMAGES_DIR : {TRAIN_IMAGES_DIR}')
print(f'  Tập test         : {len(test_df):,} dòng | Cột submission: {SUBMISSION_COLS}')
print(f'  OCR: Paddle det + VietOCR | layout bù ML khi prominence>={PIPELINE_LAYOUT_MIN_PROMINENCE} | học động>{DYNAMIC_PROMINENCE_THRESHOLD}')
if train_labels_df is not None:
    print(f'  Train            : {len(train_labels_df):,} dòng | brand={BRAND_COL!r} product={PRODUCT_COL!r} ocr={OCR_COL!r}')
    if not HAS_BRAND_COL:
        print('                     (Không có cột brand_name -> suy brand từ nhãn/regex - chế độ legacy)')
else:
    print('  Train            : KHÔNG tìm thấy train_labels.csv')
print("✓ Đã chạy xong cell 2")

In [ ]:
# ============================================================
# CELL 3 — LỚP QUY TẮC / REGEX THƯƠNG HIỆU (have_train)
# Gồm: chuẩn hoá Unicode, lọc nhiễu mạng xã hội, 46 quy tắc regex
# thương hiệu, tách brand/product, registry thương hiệu (mở rộng động),
# và chuẩn hoá tên brand/product. Đây là phần "chuẩn cuộc thi".
# ============================================================

# ==================== TIỆN ÍCH CHUẨN HOÁ ====================

def _fold_ascii(s):
    s = str(s).replace('\u0111','d').replace('\u0110','D').casefold()
    s = unicodedata.normalize('NFD', s)
    return ''.join(c for c in s if unicodedata.category(c)!='Mn')

def strip_diacritics(text):
    return _fold_ascii(text)


# ==================== LỌC NHIỄU MẠNG XÃ HỘI ====================
SOCIAL_NOISE_WORDS = frozenset({
    'tiktok','capcut','instagram','reels','facebook','youtube',
    'follow','like','share','comment','subscribe','livestream',
    'fyp','duet','stitch','viral','trending','video','clip','news',
})

OCR_TYPO_MAP = (
    (r'canfuc([o0])', r'canfoc\1'),
    (r'vinamill\b', 'vinamilk'),
    (r'vinamik\b',  'vinamilk'),
    (r'halong\b',   'ha long'),
    (r'pat\u00ea',  'pate'),
)

SOCIAL_CAPTION_RES = (
    r'(?i)\btr[aả]\s*l[oời]\s*h[iì]nh\s*lu[aậ]n\b',
    r'(?i)\bch[oố]t\s*li[eề]n\b',
    r'(?i)\br[eẻ]\s*qu[aá]\b',
    r'(?i)\bc[aầ]m\s*đ[uự]ợc\s*tr[eê]n\s*tay\b',
    r'(?i)\binbox\b|\bib\b|\bzalo\b',
    r'(?i)\bfreeship\b|\bship\s*cod\b',
    r'(?i)\bcomment\b|\breply\b',
)

_SOCIAL_CAPTION_WORDS = frozenset({
    'tra', 'loi', 'trả', 'lời', 'hình', 'luận', 'chốt', 'liền', 'rẻ', 'quá',
    'cầm', 'tay', 'nêng', 'thúy', 'thuy', 'comment', 'reply', 'inbox', 'zalo',
})

# Tin tức / chiến dịch y tế / bài viết mẹ bé — không phải nhãn sản phẩm
INFORMATIONAL_NOISE_RES = (
    r'(?i)\bvitamin\s*a\b',
    r'(?i)\bquoc\s*te\s*thieu\s*nhi\b',
    r'(?i)\bthi\s*luc\b|\bthị\s*lực\b',
    r'(?i)\bcac\s*mom\b|\bcác\s*mom\b',
    r'(?i)\bdung\s*quen\b|\bđừng\s*quên\b',
    r'(?i)\bngay\s*mai\b|\bngày\s*mai\b',
    r'(?i)\bco\s*so\s*y\s*te\b|\bcơ\s*sở\s*y\s*tế\b',
    r'(?i)\bmien\s*phi\b|\bmiễn\s*phí\b',
    r'(?i)\bem\s*sua\s*\d+\s*thang\b|\bEM\s*SỮA\s*\d+',
    r'(?i)\btram\s*trinh\b|\btrạm\s*trình\b',
    r'(?i)\buong\s*nghen\b|\buống\s*nghen\b',
    r'(?i)\bphat\s*trien\s*xuong\b|\bphát\s*triển\s*xương\b',
    r'(?i)\b6\s*-\s*36\s*thang\b',
    r'(?i)\btang\s*cuong\b|\btăng\s*cường\b',
    r'(?i)\bcho\s*cac\s*be\b|\bcho\s*các\s*bé\b',
    r'(?i)\bsap\s*xep\s*thoi\s*gian\b|\bsắp\s*xếp\s*thời\s*gian\b',
)

_INFORMATIONAL_NOISE_WORDS = frozenset({
    'vitamin', 'mom', 'moms', 'nghen', 'thieu', 'nhi', 'quoc', 'te',
    'mien', 'phi', 'thang', 'uong', 'ngay', 'mai', 'quen', 'dung',
    'yte', 'tế', 'sua', 'em', 'be', 'bé', 'cac', 'các', 'tang', 'cuong',
    'cường', 'xuong', 'xương', 'thiluc', 'thịlực', 'trinh', 'tram',
})

# Token ngắn / từ thường gặp — không match sub-line product trong BRAND_RULES
_PRODUCT_SUBLINE_BLOCKLIST = frozenset({
    'mom', 'em', 'be', 'cac', 'lon', 'can', 'hu', 'new', 'gold', 'plus',
    'den', 'sen', 'do', 'ha', 'long', 'cot', 'hop', 'sua', 'ml', 'g', 'kg',
})


def _is_informational_noise(text):
    """Tin tức / chiến dịch / caption bài viết — không phải brand/product."""
    if not text or not str(text).strip():
        return False
    raw = unicodedata.normalize('NFC', str(text).strip())
    if extract_by_rules(raw) or detect_brand_in_ocr(raw):
        return False
    t = _fold_ascii(raw)
    if any(re.search(pat, t) for pat, _ in _PRODUCT_LINE_PATTERNS):
        return False
    t_low = raw.lower()
    if any(re.search(p, t_low) for p in SOCIAL_CAPTION_RES):
        return True
    t = _fold_ascii(raw)
    if any(re.search(p, t_low) or re.search(p, t) for p in INFORMATIONAL_NOISE_RES):
        return True
    words = re.findall(r'[^\W\d_]+', t_low, flags=re.UNICODE)
    if len(words) >= 4:
        hits = sum(1 for w in words if w in _INFORMATIONAL_NOISE_WORDS)
        if hits >= 2:
            return True
    if re.search(r'(?i)\b(19|20)\d{2}\b', raw) and re.search(r'(?i)thieu\s*nhi|quoc\s*te', t):
        return True
    if len(words) >= 10:
        has_brand = bool(extract_by_rules(raw) or detect_brand_in_ocr(raw))
        has_prod = any(re.search(pat, t) for pat, _ in _PRODUCT_LINE_PATTERNS)
        if not has_brand and not has_prod:
            if _is_description_prose(raw) or hits >= 1:
                return True
    return False


def _is_plausible_brand_name(text):
    """Brand hợp lệ — loại headline tin tức / chiến dịch."""
    if not text or _is_informational_noise(text):
        return False
    if _is_description_prose(text) or _is_glued_product_brand(text):
        return False
    words = str(text).split()
    if len(words) > 6:
        return False
    folded = _fold_ascii(text)
    if extract_by_rules(text):
        return True
    if detect_brand_in_ocr(text):
        return True
    for brand in KNOWN_BRANDS:
        if _fold_ascii(brand) == folded:
            return True
    if folded in BRAND_REGISTRY:
        return True
    if globals().get('_brand_support_count'):
        try:
            if _brand_support_count(text) >= MIN_BRAND_SUPPORT:
                return True
        except Exception:
            pass
    if re.search(r'(?i)\b(19|20)\d{2}\b', text):
        return False
    if len(words) <= 2:
        return True
    return False


_PRODUCT_LINE_PATTERNS = [
    (r'(?i)\bion\s+pediasure(?:\s+uc|\s+úc)?\b', 'Ion Pediasure Úc'),
    (r'(?i)\bpediasure\b', 'PediaSure'),
    (r'(?i)\bsimilac\b', 'Similac'),
    (r'(?i)\bglucerna\b', 'Glucerna'),
    (r'(?i)\bensure(?:\s+gold)?\b', 'Ensure'),
    (r'(?i)\bprofutura\b', 'Profutura'),
    (r'(?i)\bdielac\b', 'Dielac'),
    (r'(?i)\bnan\s+optipro\b', 'NAN OPTIpro'),
    (r'(?i)\bnan\b', 'NAN'),
    (r'(?i)\bcombiotic\b', 'Combiotic'),
    (r'(?i)\bgrowplus\b', 'GrowPlus'),
    (r'(?i)\bcolos\s*baby\b', 'Colos Baby'),
    (r'(?i)\btra\s+sen\s+vang\b', 'Trà Sen Vàng'),
    (r'(?i)\bpate\s+c[ộo]t\s*d[eè]n\b', 'Pate Cột Đèn'),
]


def _is_social_caption(text):
    """Caption/reply TikTok — không phải brand/product."""
    if not text or not str(text).strip():
        return False
    t_low = unicodedata.normalize('NFC', str(text).strip()).lower()
    if any(re.search(p, t_low) for p in SOCIAL_CAPTION_RES):
        return True
    words = re.findall(r'[^\W\d_]+', t_low, flags=re.UNICODE)
    if len(words) >= 5:
        hits = sum(1 for w in words if w in _SOCIAL_CAPTION_WORDS)
        if hits >= 2:
            return True
    if len(words) > 8:
        prod_hint = r'pediasure|similac|ensure|nan|dielac|vinamilk|milo|aptamil|hipp|friso|pate'
        if not re.search(prod_hint, t_low, re.I):
            return True
    return False


def _strip_social_caption_segments(text):
    """Cắt đoạn caption/reply trong OCR trước predict."""
    t = str(text or '').strip()
    if not t:
        return t
    anchors = (
        r'abbott|vinamilk|nestl|pediasure|similac|ensure|aptamil|hipp|friso|nan|'
        r'dielac|canfoco|halong|highlands|pate|vinamill'
    )
    t = re.sub(
        rf'(?i)^(?:tr[aả]\s*l[oời]|reply|comment)[^.]*?(?=\b(?:{anchors})\b)',
        ' ', t)
    t = re.sub(
        r'(?i)\btr[aả]\s*l[oời]\s*h[iì]nh\s*lu[aậ]n\b[^.]*?(?=\b(?:abbott|vinamilk|pediasure|similac)\b)',
        ' ', t)
    t = re.sub(
        r'(?i)\bch[oố]t\s+li[eề]n\b[^.]*?(?=\b(?:pediasure|similac|ensure|abbott)\b)',
        ' ', t)
    t = re.sub(r'(?i)\bc[aầ]m\s+đ[uự]ợc\s+tr[eê]n\s+tay\b', ' ', t)
    t = re.sub(r'(?i)\br[eẻ]\s+qu[aá]\b', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()


def clean_social_ocr(text):
    text = re.sub(r'@[\w\.]+|#\w+|https?://\S+|www\.\S+', ' ', str(text))
    text = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', ' ', text)
    text = re.sub(r'(\d\s*){5,}', ' ', text)
    for pat, repl in OCR_TYPO_MAP:
        text = re.sub(pat, repl, text, flags=re.IGNORECASE)
    text = _strip_social_caption_segments(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


PRODUCT_MIN_OCR_CHARS = 4

def _is_ocr_noise_only(text):
    text = str(text or '').strip()
    if len(text) < PRODUCT_MIN_OCR_CHARS: return True
    tokens = re.sub(
        r'[^\w\s\+\u00c0-\u1ef9\u0111\u0110]', ' ', text.lower()
    ).split()
    if not tokens: return True
    meaningful = [
        t for t in tokens
        if len(t)>=3 and t not in SOCIAL_NOISE_WORDS
        and not t.startswith('@') and 'www' not in t
    ]
    return len(meaningful)==0


# ==================== QUY TẮC THƯƠNG HIỆU v4.0 ====================
# Format: (regex, canonical_brand, [product_line_keywords])
BRAND_RULES = [
    (r'ha\s*long\s*canf[ou]c[o0].*pate.*c[\u1ed9o]t|c[\u1ed9o]t\s*[d\u0111][\u00e8e]n.*ha\s*long\s*canf[ou]c[o0]',
     'Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n', []),
    (r'ha\s*long\s*canf[ou]c[o0].*pate|canf[ou]c[o0].*pate\s*c[\u1ed9o]t|pate\s*c[\u1ed9o]t.*canf[ou]c[o0]',
     'Ha Long Canfoco Pate', []),
    (r'ha\s*long\s*canf[ou]c[o0]|halong\s*canf[ou]c[o0]|canfuc[o0]|canfood|\bcanf[ou]c[o0]\b',
     'Ha Long Canfoco', []),
    (r'halong\s*canf\b|ha\s*long\s*canf\b', 'Ha Long Canfoco', []),

    (r'[d\u0111][\u1ed3o]\s*h[\u1ed9o]p\s*h[\u1ea1a]\s*long|do\s*hop\s*ha\s*long',
     '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long', []),
    (r'hophalong|hop\s*ha\s*long', '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long', []),

    (r'pate\s*c[\u1ed9o]t\s*[d\u0111][\u00e8e]n|pate\s*cot\s*den|c[\u1ed9o]t\s*[d\u0111][\u00e8e]n\s*h[\u1ea3a]i\s*ph[\u00f2o]ng',
     'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng', []),
    (r'cotc[e]n.*hai|cot\s*c[e]n.*hai|cotd.*hai\s*ph[\u00f2o]ng',
     'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng', []),
    (r'[\[fjh]?at[e\u00ea].*cot.*d[e\u00ea\u00e8n]|cotc[e]n|cot\s*c[e]n\b|\bcotd[e\u00ea]?\b',
     'Pate C\u1ed9t \u0110\u00e8n', []),

    (r'pate\s*h[\u1ea1a]\s*long|h[\u1ea1a]\s*long\s*pate', 'Ha Long Canfoco Pate', []),
    (r'vinamilk|vinamill|vinamik', 'Vinamilk',
     ['flex','adm gold','sure','canxi','colosbaby','colos baby','ong tho','\u00f4ng th\u1ecd','dielac','grow']),
    (r'th\s*true|thtrue|th\s*true\s*milk', 'TH True Milk',
     ['true yogurt','grow','school milk','true butter']),
    (r'dutch\s*lady|c\u00f4\s*g\u00e1i|dutchlady', 'Dutch Lady',
     ['grow','complete','canxi','mom']),
    (r'nutifood|\bnuti\b', 'Nutifood', ['growplus','grow plus','pedia','iq']),
    (r'\bensure\b', 'Abbott Ensure', ['gold','original','plus']),
    (r'pediasure', 'Abbott PediaSure', []),
    (r'similac', 'Abbott Similac', []),
    (r'glucerna', 'Abbott Glucerna', []),
    (r'\bmilo\b', 'Nestl\u00e9 Milo', []),
    (r'nestle|nestl\u00e9', 'Nestl\u00e9',
     ['milo','coffee mate','carnation','nestea','nan','s\u1eefa b\u1ed9t']),
    (r'aptamil', 'Aptamil', ['first infant formula','profutura','gold']),
    (r'\bhipp\b', 'HiPP', ['combiotic','organic']),
    (r'\bfriso\b', 'Friso', ['gold','comfort','prestige']),
    (r'\bmeiji\b', 'Meiji', ['growing','step']),
    (r'nan\s*(optipro|opti\s*pro|infinipro|infini\s*pro|supremepro|supreme\s*pro|a2|ha)\b',
     'Nestl\u00e9 NAN', []),
    (r'sua\s*nan\b|nestle\s*nan|nestl\u00e9\s*nan', 'Nestl\u00e9 NAN', []),
    (r'\bdulac\b|\bdielac\b', 'Vinamilk Dielac', []),
    (r'ba\s*vi\b|bavi\b|ba\s*v\u00ec', 'Ba V\u00ec', ['gold']),
    (r'lothamilk', 'Lothamilk', ['canxi']),
    (r'yomost', 'Yomost', []),
    (r'dalat\s*milk|\u0111\u00e0\s*l\u1ea1t', '\u0110\u00e0 L\u1ea1t Milk', []),
    (r'\bkun\b|kun\s*milk', 'Kun', ['chocolate','strawberry']),
    (r'\bfami\b', 'Fami', ['canxi','kid']),
    (r'anlene', 'Anlene', ['gold','concentrate']),
    (r'\banchor\b', 'Anchor', ['butter','cream']),
    (r'vissan', 'Vissan',
     ['pate heo','pate ga','pate g\u00e0','xuc xich','x\u00fac x\u00edch','lap xuong','l\u1ea1p x\u01b0\u1edfng']),
    (r'\bhafi\b', 'Hafi', ['pate']),
    (r'ba\s*huan|ba\s*hu\u00e2n', 'Ba Hu\u00e2n', ['pate']),
    (r'san\s*ha\b|san\s*h\u00e0', 'San H\u00e0', ['pate']),
    (r'\bcp\b|c\.p\.', 'CP', ['pate','x\u00fac x\u00edch']),
    (r'long\s*bien|long\s*bi\u00ean', 'Long Bi\u00ean', ['pate']),
    (r'highlands?\s*coffee', 'Highlands Coffee',
     ['tra sen vang','tra vai','banh mi que','americano']),
    (r'the\s*coffee\s*house', 'The Coffee House', ['tra phuc kien']),
    (r'nhan\s*hoa\s*foods?', 'Nh\u00e2n H\u00f2a Foods', ['pate']),
    (r'\bacnes\b', 'Acnes', ['vitamin cleanser']),
    (r'\bpate\b|pat\u00ea', 'Pate', []),
]

_SUB_NAMES = {
    'flex':'Flex','adm gold':'ADM Gold','sure':'Sure','canxi':'Canxi',
    'colosbaby':'ColosBaby','colos baby':'Colos Baby',
    'ong tho':'\u00d4ng Th\u1ecd','\u00f4ng th\u1ecd':'\u00d4ng Th\u1ecd',
    'dielac':'Dielac','grow':'Grow','true yogurt':'True Yogurt',
    'school milk':'School Milk','true butter':'True Butter',
    'growplus':'GrowPlus+','grow plus':'GrowPlus+','pedia':'Pedia','iq':'IQ',
    'gold':'Gold','original':'Original','plus':'Plus',
    'milo':'Milo','coffee mate':'Coffee Mate','nan':'NAN','s\u1eefa b\u1ed9t':'S\u1eefa B\u1ed9t',
    'first infant formula':'First Infant Formula','profutura':'Profutura',
    'combiotic':'Combiotic','organic':'Organic',
    'growing':'Growing','step':'Step',
    'comfort':'Comfort','prestige':'Prestige',
    'pate heo':'Pate Heo','pate ga':'Pate G\u00e0','pate g\u00e0':'Pate G\u00e0',
    'xuc xich':'X\u00fac X\u00edch','x\u00fac x\u00edch':'X\u00fac X\u00edch',
    'lap xuong':'L\u1ea1p X\u01b0\u1edfng','l\u1ea1p x\u01b0\u1edfng':'L\u1ea1p X\u01b0\u1edfng',
    'pate':'Pate',
    'tra sen vang':'Tr\u00e0 Sen V\u00e0ng','tra vai':'Tr\u00e0 V\u1ea3i',
    'banh mi que':'B\u00e1nh M\u00ec Que','americano':'Americano',
    'tra phuc kien':'Tr\u00e0 Ph\u00fac Ki\u1ebfn',
    'vitamin cleanser':'Vitamin Cleanser','concentrate':'Concentrate',
    'butter':'Butter','cream':'Cream','chocolate':'Chocolate','strawberry':'Strawberry',
    'kid':'Kid','kid+':'Kid+',
}


def _normalize_for_rules(text):
    tl = text.lower().replace('pat\u00ea','pate')
    tl = re.sub(r'vina[jilm1]{0,1}milk', 'vinamilk', tl, flags=re.IGNORECASE)
    tl = re.sub(r'canfuc([o0])', r'canfoc\1', tl, flags=re.IGNORECASE)
    tl = re.sub(r'halong\b', 'ha long', tl, flags=re.IGNORECASE)
    return tl


# --- Bộ lọc token chống ảo giác (anti-hallucination) ---
def _build_brand_vocab():
    vocab = set()
    for _pat, brand, lines in BRAND_RULES:
        for w in brand.split(): vocab.add(w.casefold())
        for line in lines:
            for w in line.title().split(): vocab.add(w.casefold())
    return frozenset(vocab)

_BRAND_VOCAB = _build_brand_vocab()
_BRAND_VOCAB_FOLDED = frozenset(_fold_ascii(t) for t in _BRAND_VOCAB)


def _ocr_word_set(text):
    tl = _normalize_for_rules(text)
    return {_fold_ascii(w) for w in re.findall(r'[^\W\d_]+', tl, flags=re.UNICODE) if w}


def _token_in_ocr(tok, ocr_text, ocr_words):
    tf = _fold_ascii(tok)
    if tf in ocr_words: return True
    compact = _fold_ascii(ocr_text).replace(' ','')
    if len(tf)>=2 and tf in compact: return True
    return any(len(tf)>=2 and tf in ow for ow in ocr_words)


def _assign_readable_brand_tokens(ocr_text, candidate):
    """Chỉ giữ token thuộc brand list VÀ đọc được từ ocr_text (chống ảo giác)."""
    if not candidate or not candidate.strip(): return ''
    ocr_words = _ocr_word_set(ocr_text)
    kept = []
    for tok in candidate.split():
        if _fold_ascii(tok) not in _BRAND_VOCAB_FOLDED: continue
        if _token_in_ocr(tok, ocr_text, ocr_words): kept.append(tok)
    return ' '.join(kept)


def extract_by_rules(text):
    # Rule neo theo regex trên OCR nên TIN CẬY -> trả về nhãn đầy đủ.
    if not text: return ''
    tl = _normalize_for_rules(text)
    matched = ''
    for pattern, brand, lines in BRAND_RULES:
        if re.search(pattern, tl, re.IGNORECASE):
            for line in lines:
                if re.search(line, tl, re.IGNORECASE):
                    sub = _SUB_NAMES.get(line, line.title())
                    matched = f'{brand} {sub}'.strip()
                    break
            if not matched:
                matched = brand
            break
    return matched


def post_process_prediction(pred, ocr_text):
    """Nâng cấp nhãn 'Pate' chung chung -> cụ thể hơn bằng cách quét lại OCR."""
    if not pred: return pred
    nd = _fold_ascii(ocr_text) if ocr_text else ''
    if pred == 'Pate':
        if re.search(r'hai\s*ph[o]ng', nd):
            if re.search(r'cot|cotcen|cotd|col\s*d', nd):
                return 'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng'
        if any(re.search(p, nd) for p in [r'cot\s*d[e]n',r'\bcotd[e]?\b',r'cotcen',r'cot\s*cen']):
            return 'Pate C\u1ed9t \u0110\u00e8n'
        if re.search(r'ha\s*long|halong', nd):
            return '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long'
    if pred == 'Nestl\u00e9':
        if re.search(r'\bnan\b|nan\s*opti|nan\s*infini|nan\s*supreme', nd):
            return 'Nestl\u00e9 NAN'
        if re.search(r'\bmilo\b', nd):
            return 'Nestl\u00e9 Milo'
    if pred == 'Vinamilk':
        if re.search(r'dulac|dielac', nd):
            return 'Vinamilk Dielac'
    return pred


# ==================== CHUẨN HOÁ TÊN SẢN PHẨM ====================
_PRODUCT_CANON_MAP = None

def _build_product_canon_map():
    """Gộp các biến thể chính tả/dấu câu của cùng 1 sản phẩm bằng fold-key,
    chọn spelling phổ biến nhất trong train data làm chính tắc."""
    global _PRODUCT_CANON_MAP
    if _PRODUCT_CANON_MAP is not None:
        return _PRODUCT_CANON_MAP
    _PRODUCT_CANON_MAP = {}
    try:
        _pcol = PRODUCT_COL if ('PRODUCT_COL' in globals() and PRODUCT_COL) else 'product_name'
        names = train_labels_df[_pcol].astype(str).str.strip()
        names = names[names != '']
        counts = Counter(unicodedata.normalize('NFC', n) for n in names)
        groups = {}
        for name, cnt in counts.items():
            key = _fold_ascii(name)
            groups.setdefault(key, []).append((cnt, name))
        for key, variants in groups.items():
            variants.sort(key=lambda x: (-x[0], -len(x[1])))
            _, best_name = variants[0]
            _PRODUCT_CANON_MAP[key] = best_name
    except Exception:
        pass
    return _PRODUCT_CANON_MAP


def _is_plausible_product_name(text):
    """Tên product hợp lệ — loại caption dài / generic."""
    if not text or _is_social_caption(text) or _is_informational_noise(text):
        return False
    if _is_generic_product(text):
        return False
    words = str(text).split()
    if len(words) > 8:
        return False
    folded = _fold_ascii(text)
    if any(re.search(pat, folded) for pat, _ in _PRODUCT_LINE_PATTERNS):
        return True
    canon = _build_product_canon_map()
    if folded in canon:
        return True
    if globals().get('_product_support_count'):
        try:
            if _product_support_count(text) >= MIN_BRAND_SUPPORT:
                return True
        except Exception:
            pass
    if _is_description_prose(text):
        return False
    if len(words) <= 3 and any(re.search(pat, folded) for pat, _ in _PRODUCT_LINE_PATTERNS):
        return True
    return False


def guess_product_from_ocr(ocr_text, brand=''):
    """Trích product từ OCR sau khi đã có brand (catalog + pattern + sub-lines rules)."""
    ocr_text = str(ocr_text or '').strip()
    if not ocr_text or _is_ocr_noise_only(ocr_text):
        return ''
    if _is_description_prose(ocr_text):
        ocr_text = clean_social_ocr(ocr_text)
    if not ocr_text:
        return ''
    tl = _normalize_for_rules(ocr_text)
    folded = _fold_ascii(tl)
    compact = folded.replace(' ', '')

    for pat, label in _PRODUCT_LINE_PATTERNS:
        if re.search(pat, tl, re.IGNORECASE):
            return label

    canon = _build_product_canon_map()
    for key, pname in sorted(canon.items(), key=lambda x: -len(x[0])):
        if len(key) < 4 or key in _PRODUCT_SUBLINE_BLOCKLIST:
            continue
        if len(key) <= 5:
            if not re.search(r'(?<!\w)' + re.escape(key) + r'(?!\w)', folded):
                continue
        elif key not in folded and key.replace(' ', '') not in compact:
            continue
        if not _is_social_caption(pname) and not _is_informational_noise(pname):
            return pname

    bf = _fold_ascii(brand) if brand else ''
    for _pat, _bname, lines in BRAND_RULES:
        if bf and bf not in _fold_ascii(_bname) and _fold_ascii(_bname) not in bf:
            if not re.search(_pat, tl, re.IGNORECASE):
                continue
        for line in lines:
            if len(line) < 4 or line.lower() in _PRODUCT_SUBLINE_BLOCKLIST:
                continue
            if not re.search(r'(?<!\w)' + re.escape(line) + r'(?!\w)', tl, re.IGNORECASE):
                continue
            sub = _SUB_NAMES.get(line, line.title())
            if not _is_generic_product(sub) and not _is_informational_noise(sub):
                return sub
    return ''


def normalize_product_name(name):
    """Chuẩn hoá nhãn sản phẩm: NFC + map catalog train; không .title() chuỗi lạ."""
    if not name:
        return ''
    name = unicodedata.normalize('NFC', str(name).strip())
    name = re.sub(r'\s+', ' ', name)
    if not name or _is_social_caption(name):
        return ''
    canon_map = _build_product_canon_map()
    key = _fold_ascii(name)
    if key in canon_map:
        return canon_map[key]
    return name


# ==================== TÁCH BRAND / PRODUCT ====================
KNOWN_BRANDS = [
    'Ha Long Canfoco', 'Pate C\u1ed9t \u0110\u00e8n', 'TH True Milk', 'Dutch Lady',
    'Highlands Coffee', 'The Coffee House', 'Nh\u00e2n H\u00f2a Foods', 'H\u1ea1 Long',
    'Abbott', 'Nestl\u00e9', 'Vinamilk', 'Nutifood', 'Aptamil', 'HiPP', 'Friso',
    'Meiji', 'Ba V\u00ec', 'Lothamilk', 'Yomost', '\u0110\u00e0 L\u1ea1t Milk', 'Kun',
    'Fami', 'Anlene', 'Anchor', 'Vissan', 'Hafi', 'Ba Hu\u00e2n', 'San H\u00e0',
    'CP', 'Long Bi\u00ean', 'Acnes',
]

# Hãng (manufacturer) vs dòng sản phẩm — ưu tiên hãng làm brand_name
_MANUFACTURER_CANON = frozenset({
    'Abbott', 'Nestlé', 'Vinamilk', 'Nutifood', 'Aptamil', 'HiPP', 'Friso', 'Meiji',
    'Ha Long Canfoco', 'TH True Milk', 'Dutch Lady', 'Highlands Coffee', 'The Coffee House',
    'Vissan', 'Dove', 'Nhân Hòa Foods', 'Đà Lạt Milk', 'Ba Vì', 'Lothamilk', 'Yomost',
    'Fami', 'Anlene', 'Anchor', 'Hafi', 'Ba Huân', 'San Hà', 'CP', 'Long Biên', 'Acnes',
    'Hạ Long', 'Pate Cột Đèn',
})
_MANUFACTURER_FOLDED = frozenset(_fold_ascii(b) for b in _MANUFACTURER_CANON)

# (product_regex_on_folded, manufacturer_regex, brand_out, product_out)
_BRAND_PRODUCT_OCR_PAIRS = [
    (r'similac', r'abbott', 'Abbott', 'Similac'),
    (r'pediasure', r'abbott', 'Abbott', 'PediaSure'),
    (r'glucerna', r'abbott', 'Abbott', 'Glucerna'),
    (r'ensure', r'abbott', 'Abbott', 'Ensure'),
    (r'\bnan\b', r'nestl', 'Nestlé', 'NAN'),
    (r'dielac', r'vinamilk', 'Vinamilk', 'Dielac'),
    (r'profutura', r'aptamil', 'Aptamil', 'Profutura'),
]

_DESCRIPTION_PROSE_MARKERS = (
    'cong thuc', 'duoc phat trien', 'được phát triển', 'nham ho tro', 'nhằm hỗ trợ',
    'tieu hoa', 'tiêu hóa', 'phat trien', 'phát triển', 'cao cap', 'cao cấp',
    'cua abbott', 'của abbott', 'hemiendich', 'tieuhoa', 'tren hoi', 'protection',
    'resentional', 'duoc phat', 'tro he', 'choi tren',
)


def _is_description_prose(text):
    """Đoạn mô tả/quảng cáo dài — không phải product_name."""
    if not text or not str(text).strip():
        return False
    raw = str(text).strip()
    t = _fold_ascii(raw)
    if _is_social_caption(raw):
        return True
    words = raw.split()
    if len(words) > 6:
        hits = sum(1 for m in _DESCRIPTION_PROSE_MARKERS if m in t or m in raw.lower())
        if hits >= 1:
            return True
        vn = ('duoc', 'được', 'nham', 'nhằm', 'cong', 'công', 'cao', 'cap', 'cấp',
              'cua', 'của', 'tren', 'trên', 'phat', 'phát', 'trien', 'triển', 'ho tro')
        if sum(1 for w in words if w.lower() in vn) >= 2:
            return True
    if ',' in raw and len(words) > 4:
        return True
    if len(raw) > 42:
        if not any(re.search(p, t) for p, _ in _PRODUCT_LINE_PATTERNS):
            return True
        desc_hits = sum(1 for m in _DESCRIPTION_PROSE_MARKERS if m in t)
        if desc_hits >= 1:
            return True
    return False


def _is_glued_product_brand(name):
    """Brand nhìn như dòng SP dính chữ OCR (vd Similac Totalladongsữa)."""
    if not name:
        return False
    f = _fold_ascii(name).replace(' ', '')
    for key in ('similac', 'pediasure', 'ensure', 'glucerna', 'profutura', 'dielac'):
        if f.startswith(key) and len(f) > len(key) + 3:
            return True
    return False


def _score_brand_candidate(canonical):
    """Điểm alias brand: ưu tiên hãng, phạt tên dài/dính rác."""
    if not canonical:
        return -999
    score = 0
    cf = _fold_ascii(canonical)
    if cf in _MANUFACTURER_FOLDED or canonical in _MANUFACTURER_CANON:
        score += 200
    if any(cf.startswith(_fold_ascii(m)) for m in _MANUFACTURER_CANON):
        score += 150
    score -= max(0, len(canonical) - 18)
    score -= max(0, len(canonical.split()) - 3) * 25
    if _is_glued_product_brand(canonical):
        score -= 300
    if _is_description_prose(canonical):
        score -= 400
    return score


def reconcile_brand_product(ocr_text, brand, product):
    """Chuẩn hoá cặp brand/product từ OCR — Abbott+Similac, bỏ mô tả dài."""
    ocr_text = str(ocr_text or '').strip()
    brand = str(brand or '').strip()
    product = str(product or '').strip()
    if not ocr_text:
        return brand, product

    folded = _fold_ascii(_normalize_for_rules(ocr_text))

    for prod_pat, manu_pat, b_out, p_out in _BRAND_PRODUCT_OCR_PAIRS:
        if re.search(prod_pat, folded, re.I) and re.search(manu_pat, folded, re.I):
            return b_out, p_out

    line_prod = guess_product_from_ocr(ocr_text, brand) or ''
    manu = ''
    if re.search(r'abbott', folded, re.I):
        manu = 'Abbott'
    elif re.search(r'nestl', folded, re.I):
        manu = 'Nestlé'
    elif re.search(r'vinamilk|vinamill', folded, re.I):
        manu = 'Vinamilk'
    elif re.search(r'aptamil', folded, re.I):
        manu = 'Aptamil'

    if _is_glued_product_brand(brand) or _is_description_prose(brand):
        brand = ''
    if _is_description_prose(product):
        product = ''
    if product and _fold_ascii(brand) == _fold_ascii(product):
        product = ''

    if not brand and manu:
        brand = manu
    if not product and line_prod:
        product = line_prod

    if brand and not product and line_prod:
        product = line_prod
    if product and not brand and manu:
        brand = manu

    if _is_glued_product_brand(brand) and line_prod:
        if manu:
            brand = manu
        product = line_prod

    brand = normalize_brand_name(brand) if brand else ''
    if product:
        product = normalize_product_name(product)
        if brand:
            bf, pf = _fold_ascii(brand), _fold_ascii(product)
            if pf.startswith(bf + ' '):
                product = product[len(brand):].strip()
        if _is_generic_product(product) or _is_description_prose(product):
            product = ''
    return brand, product


MERGED_SPLIT = {
    'Ha Long Canfoco Pate C\u1ed9t \u0110\u00e8n': ('Ha Long Canfoco', 'Pate C\u1ed9t \u0110\u00e8n'),
    'Ha Long Canfoco Pate':                       ('Ha Long Canfoco', 'Pate'),
    'Ha Long Canfoco':                            ('Ha Long Canfoco', ''),
    '\u0110\u1ed3 H\u1ed9p H\u1ea1 Long':         ('H\u1ea1 Long', '\u0110\u1ed3 H\u1ed9p'),
    'Pate C\u1ed9t \u0110\u00e8n H\u1ea3i Ph\u00f2ng': ('Pate C\u1ed9t \u0110\u00e8n', 'H\u1ea3i Ph\u00f2ng'),
    'Pate C\u1ed9t \u0110\u00e8n':                 ('Pate C\u1ed9t \u0110\u00e8n', ''),
    'Pate':                                       ('', 'Pate'),
    'Vinamilk Dielac':                            ('Vinamilk', 'Dielac'),
    'Nestl\u00e9 NAN':                            ('Nestl\u00e9', 'NAN'),
    'Nestl\u00e9 Milo':                           ('Nestl\u00e9', 'Milo'),
}

GENERIC_PRODUCT_TOKENS = frozenset({
    'cot','hop','milk','sua','gold','plus','new','lon','can','hu','ml','g','kg',
    'den','sen','do','ha','long',
})

MIN_BRAND_SUPPORT   = 2   # brand mới từ train cần >= số mẫu này
MIN_BRAND_ALIAS_LEN = 4   # alias ngắn hơn -> bỏ qua (tránh dùng 'cp','ps')

BRAND_REGISTRY = {}
def _register_brand(canonical):
    canonical = unicodedata.normalize('NFC', str(canonical).strip())
    if not canonical: return
    aliases = set()
    f = _fold_ascii(canonical)
    aliases.add(f)
    aliases.add(f.replace(' ', ''))
    for a in list(aliases):
        if len(a.replace(' ','')) >= MIN_BRAND_ALIAS_LEN or ' ' in a:
            BRAND_REGISTRY[a] = canonical

for _b in KNOWN_BRANDS:
    _register_brand(_b)

_BRAND_CANON_FOLD = {}   # fold(brand) -> canonical casing (ưu tiên train)

def build_brand_knowledge_from_train():
    """Học brand mới (vd 'Dove') từ cột brand_name của train, có kiểm soát
    support tối thiểu. Không bịa: chỉ đưa brand có trong train."""
    if not (HAS_BRAND_COL and train_labels_df is not None):
        return 0
    s = train_labels_df[BRAND_COL].astype(str).map(lambda x: unicodedata.normalize('NFC', x.strip()))
    s = s[s != '']
    vc = s.value_counts()
    added = 0
    for brand, cnt in vc.items():
        _BRAND_CANON_FOLD.setdefault(_fold_ascii(brand), brand)
        if cnt >= MIN_BRAND_SUPPORT:
            _register_brand(brand)
            added += 1
    return added


def normalize_brand_name(name):
    if not name: return ''
    name = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(name).strip()))
    if not name: return ''
    f = _fold_ascii(name)
    if f in _BRAND_CANON_FOLD: return _BRAND_CANON_FOLD[f]
    if f in BRAND_REGISTRY:    return BRAND_REGISTRY[f]
    fc = f.replace(' ', '')
    if fc in BRAND_REGISTRY:   return BRAND_REGISTRY[fc]
    return name


def detect_brand_in_ocr(ocr_text):
    """Quét alias brand trong OCR — ưu tiên hãng (manufacturer), phạt tên dính OCR."""
    if not ocr_text:
        return ''
    folded = _fold_ascii(_normalize_for_rules(ocr_text))
    words = set(folded.split())
    compact = folded.replace(' ', '')
    best, best_score = '', -10**9
    for alias, canonical in BRAND_REGISTRY.items():
        a = alias.replace(' ', '')
        if len(a) < MIN_BRAND_ALIAS_LEN:
            continue
        if ' ' in alias:
            hit = alias in folded
        else:
            hit = (a in words) or (len(a) >= 6 and a in compact)
        if not hit or _is_description_prose(canonical):
            continue
        sc = _score_brand_candidate(canonical)
        if sc > best_score:
            best_score, best = sc, canonical
    return best


def _is_generic_product(product):
    if not product: return True
    f = _fold_ascii(product).strip()
    if len(f) < 2: return True
    toks = f.split()
    if len(toks) == 1 and toks[0] in GENERIC_PRODUCT_TOKENS: return True
    return False


def split_brand_product(merged):
    """Tách nhãn gộp -> (brand_name, product_name)."""
    if not merged: return '', ''
    nm = re.sub(r'\s+', ' ', unicodedata.normalize('NFC', str(merged).strip()))
    if nm in MERGED_SPLIT:
        b, p = MERGED_SPLIT[nm]
        return normalize_brand_name(b), p
    nf = _fold_ascii(nm)
    for brand in sorted(KNOWN_BRANDS, key=len, reverse=True):
        bf = _fold_ascii(brand)
        if nf == bf:
            return normalize_brand_name(brand), ''
        if nf.startswith(bf + ' '):
            return normalize_brand_name(brand), nm[len(brand):].strip()
    return '', nm


# --- Kiểm thử nhanh quy tắc ---
_rule_tests = [
    ('HA LONG CANFOCO Pate C\u1ed9t \u0110\u00e8n', 'Ha Long Canfoco'),
    ('HALONG CANFUCO j TIkTok',                    'Ha Long Canfoco'),
    ('Vinamilk Flex 180ml',                        'Vinamilk'),
    ('MILO Nestle chocolate 3 in 1',               'Nestl\u00e9'),
    ('tiktok capcut viral',                        ''),
]
_ok = sum(1 for t, e in _rule_tests
          if (e == '' and extract_by_rules(t) == '') or (e != '' and e.lower() in extract_by_rules(t).lower()))
print(f'Lớp quy tắc: {len(BRAND_RULES)} regex | Kiểm thử nhanh: {_ok}/{len(_rule_tests)} đạt')
print("✓ Đã chạy xong cell 3")

In [ ]:
# ============================================================
# CELL 4 — LÕI PIPELINE (BASELINE / Ý TƯỞNG CHỦ ĐẠO)
# Phát hiện khung vật thể (đỏ) + khung chữ, đọc bằng VietOCR (cell 7),
# hậu xử lý song ngữ, rồi trích xuất brand/product theo bố cục.
# Import cv2: nếu lệch ABI với numpy hiện tại -> cài lại opencv-headless (không đổi numpy).
# ============================================================
import sys, subprocess

def _import_cv2_safe():
    try:
        import cv2 as _cv2
        return _cv2
    except (ImportError, RuntimeError, OSError):
        print("  OpenCV lệch ABI numpy %s -> cài lại opencv-python-headless..." % (
            __import__('numpy').__version__))
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y",
             "opencv-python", "opencv-contrib-python", "opencv-python-headless"],
            check=False,
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
             "--force-reinstall", "opencv-python-headless>=4.10.0"],
            check=False,
        )
        for _m in list(sys.modules):
            if _m == "cv2" or _m.startswith("cv2."):
                del sys.modules[_m]
        import cv2 as _cv2
        return _cv2

cv2 = _import_cv2_safe()
import difflib

# ---------- Hậu xử lý chính tả song ngữ (Việt - Anh) ----------
ENGLISH_PRODUCT_WORDS = {
    "setup","install","gpu","cpu","model","data","image","file",
    "download","link","process","python","import","from","train",
    "test","run","error","api","version","config","github","project",
    "maybelline","mascara","hyper","curl","waterproof","gel","volume",
    "matte","cream","lotion","serum","cleanser","shampoo","conditioner",
    "eyeliner","eyebrow","lipstick","lipbalm","foundation","powder",
    "blush","skincare","active","formula","natural","organic","extract",
    "original","premium","classic","style","design","brand","product",
    "made","in","vietnam","usa","korea","japan","quality","best",
    "super","ultra","mega","extreme","pro","max","smart","light","dark",
    "color","black","white","red","blue","green","ml","g","kg","size",
    "pack","set","limit","edition","free","soft","smooth","fresh","cool",
    "dry","wet","hot","warm","sunblock","sunscreen","uv","protect","moisture",
}

def remove_vietnamese_accents(input_str):
    nfd_form = unicodedata.normalize('NFD', input_str)
    only_ascii = "".join([c for c in nfd_form if not unicodedata.combining(c)])
    return only_ascii.replace('\u0110', 'D').replace('\u0111', 'd')

def clean_bilingual_word(word):
    """Sửa lỗi từ ngữ dựa trên bộ từ điển bao bì thu nhỏ."""
    cleaned = word
    clean_word = re.sub(r'^\W+|\W+$', '', word)
    if re.match(r'^[a-zA-Z0-9]+$', clean_word):
        temp = clean_word.replace('0', 'o').replace('1', 'i')
        if temp.lower() in ENGLISH_PRODUCT_WORDS:
            return word.replace(clean_word, temp)
    if any(char in '\u00e0\u00e1\u1ea3\u00e3\u1ea1\u1eb1\u1eaf\u1eb3\u1eb5\u1eb7\u1ea7\u1ea5\u1ea9\u1eab\u1ead\u00e8\u00e9\u1ebb\u1ebd\u1eb9\u1ec1\u1ebf\u1ec3\u1ec5\u1ec7\u00ec\u00ed\u1ec9\u0129\u1ecb\u00f2\u00f3\u1ecf\u00f5\u1ecd\u1ed3\u1ed1\u1ed5\u1ed7\u1ed9\u1edd\u1edb\u1edf\u1ee1\u1ee3\u00f9\u00fa\u1ee7\u0169\u1ee5\u1eeb\u1ee9\u1eed\u1eef\u1ef1\u1ef3\u00fd\u1ef7\u1ef9\u1ef5\u0111' for char in clean_word.lower()):
        no_accent_word = remove_vietnamese_accents(clean_word)
        if no_accent_word.lower() in ENGLISH_PRODUCT_WORDS:
            if clean_word.istitle():
                final_word = no_accent_word.capitalize()
            elif clean_word.isupper():
                final_word = no_accent_word.upper()
            else:
                final_word = no_accent_word.lower()
            return word.replace(clean_word, final_word)
    return cleaned

def bilingual_post_processor(text_list):
    processed_list = []
    for text in text_list:
        if not text:
            processed_list.append("")
            continue
        text = unicodedata.normalize('NFC', text)
        words_in_line = text.split()
        cleaned_words = [clean_bilingual_word(w) for w in words_in_line]
        processed_line = " ".join(cleaned_words)
        processed_line = re.sub(r'\s+([.,!?;:])', r'\1', processed_line)
        processed_list.append(processed_line)
    return processed_list


# ---------- Từ điển & mẫu hỗ trợ trích xuất bố cục ----------
BRAND_NORMALIZATION_MAP = {
    "ha long": "Ha Long Canfoco",
    "canfoco": "Ha Long Canfoco",
    "\u0111\u1ed3 h\u1ed9p h\u1ea1 long": "Ha Long Canfoco",
    "dove": "Dove",
    "vinamilk": "Vinamilk",
    "vissan": "Vissan",
    "nestle": "Nestl\u00e9",
    "nescafe": "Nescafe",
    "omo": "Omo",
    "knorr": "Knorr",
    "maybelline": "Maybelline",
}

PROMO_KEYWORDS = {"ch\u00ednh h\u00e3ng", "\u0111\u1ed9c quy\u1ec1n", "freeship", "cam k\u1ebft", "uy t\u00edn", "qu\u00e0 t\u1eb7ng", "m\u1edbi", "new", "official"}

DESCRIPTION_INDICATORS = [
    "\u0111\u1ea7u c\u1ecd", "cho mi", "d\u1ea1ng gel", "c\u00f4ng th\u1ee9c", "ph\u1ee7 hi\u1ec7u", "h\u00e0ng mi",
    "th\u1ea5m n\u01b0\u1edbc", "ch\u1ea3i t\u1eebng", "nh\u1eb9 m\u01b0\u1ee3t", "ch\u1ed1ng tr\u00f4i", "cong v\u00fat", "su\u1ed1t nhi\u1ec1u",
    "si\u00eau m\u1ecbn", "b\u00e0o m\u00f2n", "kh\u00f4ng lo", "m\u1ec1m m\u1ecbn", "l\u00e0n da", "d\u01b0\u1ee1ng \u1ea9m",
]

VARIANT_PATTERN = r'\d+\s*(g|ml|kg|l|gr|gx|h\u1ed9p|g\u00f3i|chai|l\u1ed1c|lon|c\u00e1i|vi\u00ean|h\u0169)\s*(x\s*\d+)?|\b(combo|l\u1ed1c|v\u1ec9|set)\s*\d+\s*(h\u1ed9p|g\u00f3i|chai|l\u1ed1c|lon|c\u00e1i|vi\u00ean|h\u0169)?'
PROMO_PATTERN = r'\b(gi\u1ea3m|giam)\s*\d+%\b|\b\d+%\s*(off|gi\u1ea3m|giam)?\b|\bmua\s*\d+\s*t\u1eb7ng\s*\d+\b'


def fuzzy_match_brand(ocr_text, normalization_map, threshold=0.75):
    ocr_clean = re.sub(r'\W+', '', ocr_text.lower())
    for variant, canonical in normalization_map.items():
        v_clean = re.sub(r'\W+', '', variant.lower())
        if v_clean in ocr_clean:
            return True, canonical
    words = ocr_text.lower().split()
    for word in words:
        clean_word = re.sub(r'\W+', '', word)
        for variant, canonical in normalization_map.items():
            v_clean = re.sub(r'\W+', '', variant.lower())
            ratio = difflib.SequenceMatcher(None, clean_word, v_clean).ratio()
            if ratio >= threshold:
                return True, canonical
    return False, None


def is_negative_line(text):
    text_lower = text.lower()
    patterns = [
        r'\d{2,4}[-/\.]\d{2}[-/\.]\d{2,4}',
        r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',
        r'www\.|http[s]?://|\.com|\.vn',
        r'^\d+$',
    ]
    if any(re.search(p, text) for p in patterns):
        return True
    neg_keywords = {"ng\u00e0y s\u1ea3n xu\u1ea5t", "th\u00e0nh ph\u1ea7n", "ingredients", "ch\u1ec9 ti\u00eau", "nh\u00e2n qu\u1ea3", "di s\u1ea3n", "nh\u1eafn nh\u1ee7"}
    if any(kw in text_lower for kw in neg_keywords):
        return True
    return False


def clean_title_noise(text):
    cleaned = re.sub(r'^[A-Za-z]\s+|\s+[A-Za-z]$', '', text)
    return cleaned.strip()


def is_brand_duplicate(text, brand_name):
    if not brand_name:
        return False
    t_clean = re.sub(r'\W+', '', text.lower())
    b_clean = re.sub(r'\W+', '', brand_name.lower())
    if t_clean == b_clean:
        return True
    brand_words = [re.sub(r'\W+', '', w) for w in brand_name.lower().split()]
    if t_clean in brand_words and len(t_clean) > 1:
        return True
    return False


def is_description_line(text, y_center, max_y):
    text_lower = text.lower()
    if any(indicator in text_lower for indicator in DESCRIPTION_INDICATORS):
        return True
    if any(text_lower.startswith(word) for word in ["\u0111\u1ea7u", "d\u1ea1ng", "c\u00f4ng", "cho", "th\u1ea5m", "ph\u1ee7", "h\u00e0ng", "l\u00e0m", "gi\u00fap", "h\u1ea1t", "kh\u00f4ng"]):
        return True
    if max_y > 0 and y_center > (max_y * 0.35):
        if "smoothie" not in text_lower and "mascara" not in text_lower:
            return True
    return False


# ---------- Chấm điểm ứng viên & trích xuất thông tin ----------
def check_inside_red_box(box_coords, red_box):
    if not red_box: return 0.0
    xs, ys = [p[0] for p in box_coords], [p[1] for p in box_coords]
    xmin, xmax, ymin, ymax = min(xs), max(xs), min(ys), max(ys)
    ixmin, iymin = max(xmin, red_box[0]), max(ymin, red_box[1])
    ixmax, iymax = min(xmax, red_box[2]), min(ymax, red_box[3])
    inter_area = max(0, ixmax - ixmin) * max(0, iymax - iymin)
    box_area = (xmax - xmin) * (ymax - ymin)
    return inter_area / (box_area + 1e-5)

def compute_candidate_score(box_coords, text, max_y, red_box, img_area):
    xs, ys = [p[0] for p in box_coords], [p[1] for p in box_coords]
    height, width = max(ys) - min(ys), max(xs) - min(xs)
    y_center = min(ys) + (height / 2)
    box_area = width * height
    aspect_ratio = width / (height + 1e-5)
    position_weight = 1.0 - (y_center / max_y) if max_y > 0 else 0.5
    base_score = height * aspect_ratio * position_weight
    area_ratio = box_area / img_area if img_area > 0 else 0
    overlap_ratio = check_inside_red_box(box_coords, red_box)
    multiplier = 1.0
    if overlap_ratio > 0.5: multiplier *= 1.5
    else:
        if area_ratio > 0.03: multiplier *= 2.0
        else: multiplier *= 0.5
    return base_score * multiplier

def compute_brand_prominence_score(line, max_y):
    text, height, y_center = line["text"], line["height"], line["y_center"]
    capital_multiplier = 2.0 if text.isupper() else (1.5 if text and text[0].isupper() else 1.0)
    word_count = len(text.split())
    length_multiplier = 2.0 if 1 <= word_count <= 3 else (0.3 if word_count > 6 else 1.0)
    y_factor = (y_center / max_y) if max_y > 0 else 0.5
    position_weight = (1.0 - y_factor) if y_center < (max_y * 0.65) else 0.05
    return height * capital_multiplier * length_multiplier * position_weight


def extract_universal_product_info_v5(boxes, texts, img_shape=None, red_box=None):
    lines_info = []
    for idx, (box, text) in enumerate(zip(boxes, texts)):
        ys = [p[1] for p in box]
        height = max(ys) - min(ys)
        y_center = min(ys) + (height / 2)
        lines_info.append({"text": text, "y_center": y_center, "height": height,
                           "is_brand": False, "is_promo": False, "is_variant": False, "box": box})

    if not lines_info: return {}

    if img_shape is not None:
        img_area = img_shape[0] * img_shape[1]
        max_box_area = max((max(p[0] for p in b) - min(p[0] for p in b)) * (max(p[1] for p in b) - min(p[1] for p in b)) for b in boxes)
        if (max_box_area / img_area) < 0.010:
            return {"brand_name": " ", "product_name": " ", "variant": "Kh\u00f4ng c\u00f3", "description": ["\u1ea2nh r\u00e1c/Ch\u1eef qu\u00e1 nh\u1ecf"]}

    max_y = max(line["y_center"] for line in lines_info)
    brand_name, product_name = None, None
    product_name_parts = []

    variant_specs = []
    for line in lines_info:
        t_low = line["text"].lower()
        if (_is_social_caption(line["text"]) or _is_description_prose(line["text"])
                or _is_informational_noise(line["text"])
                or is_negative_line(line["text"])
                or any(k in t_low for k in PROMO_KEYWORDS) or re.search(PROMO_PATTERN, t_low)):
            line["is_promo"] = True
        if re.search(VARIANT_PATTERN, t_low):
            line["is_variant"] = True
            variant_specs.append(line["text"])

    valid_lines = [l for l in lines_info if not l['is_promo'] and not l['is_variant'] and len(l['text']) >= 3]
    repeating_groups = []

    for l in valid_lines:
        matched = False
        t_clean = re.sub(r'\W+', '', l['text'].lower())
        for g in repeating_groups:
            if difflib.SequenceMatcher(None, t_clean, g['clean_rep']).ratio() > 0.75:
                g['lines'].append(l)
                curr_text = l['text']
                rep_text = g['rep']
                curr_score = (len(curr_text.replace(" ", "")), -curr_text.count(" "))
                rep_score = (len(rep_text.replace(" ", "")), -rep_text.count(" "))
                if curr_score > rep_score:
                    g['rep'] = curr_text
                    g['clean_rep'] = t_clean
                matched = True
                break
        if not matched:
            repeating_groups.append({'rep': l['text'], 'clean_rep': t_clean, 'lines': [l]})

    multi_groups = [g for g in repeating_groups if len(g['lines']) >= 2]

    if multi_groups:
        for g in multi_groups:
            g['score'] = sum(compute_brand_prominence_score(l, max_y) for l in g['lines']) / len(g['lines'])
        multi_groups.sort(key=lambda x: x['score'], reverse=True)
        if len(multi_groups) == 1:
            brand_name = multi_groups[0]['rep']
            for l in multi_groups[0]['lines']: l['is_brand'] = True
        else:
            g1, g2 = multi_groups[0], multi_groups[1]
            if len(g1['rep']) > len(g2['rep']):
                product_name_parts.append(g1['rep'])
                brand_name = g2['rep']
            else:
                product_name_parts.append(g2['rep'])
                brand_name = g1['rep']
            for l in g1['lines'] + g2['lines']: l['is_brand'] = True

    if not brand_name:
        for line in lines_info:
            matched, m_brand = fuzzy_match_brand(line["text"], BRAND_NORMALIZATION_MAP)
            if matched:
                brand_name = m_brand
                line["is_brand"] = True
                break
        if not brand_name:
            for line in lines_info: line["prominence_score"] = compute_brand_prominence_score(line, max_y)
            fallback_brand = max(valid_lines, key=lambda x: x.get("prominence_score", 0), default=None)
            if fallback_brand and fallback_brand.get("prominence_score", 0) >= PIPELINE_LAYOUT_FALLBACK_BRAND:
                brand_name = fallback_brand["text"]
                fallback_brand["is_brand"] = True

    img_area = img_shape[0] * img_shape[1] if img_shape else 1.0
    product_candidates = [l for l in valid_lines if not l["is_brand"] and not is_brand_duplicate(l["text"], brand_name)]

    if not product_name_parts and product_candidates:
        for c in product_candidates:
            c["score"] = compute_candidate_score(c["box"], c["text"], max_y, red_box, img_area)
        max_score = max(c["score"] for c in product_candidates)
        for c in product_candidates:
            if c["score"] >= max_score * 0.35 and not is_description_line(c["text"], c["y_center"], max_y):
                product_name_parts.append(clean_title_noise(c["text"]))

    description_parts = [l["text"] for l in valid_lines if not l["is_brand"] and clean_title_noise(l["text"]) not in product_name_parts]
    descriptions, current_desc = [], ""
    for desc in description_parts:
        if current_desc and (desc[0].islower() or desc.startswith("kh\u00f4ng") or desc.startswith("h\u01b0\u01a1ng")):
            current_desc += " " + desc
        else:
            if current_desc: descriptions.append(current_desc)
            current_desc = desc
    if current_desc: descriptions.append(current_desc)

    return {
        "brand_name": brand_name if brand_name else " ",
        "product_name": " ".join(product_name_parts) if product_name_parts else " ",
        "variant": ", ".join(variant_specs) if variant_specs else "Kh\u00f4ng c\u00f3",
        "description": descriptions
    }


# ---------- Cắt khung, tìm khung đỏ, đọc VietOCR, gói pipeline ----------
def crop_box(pil_img, box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]
    xmin, xmax = int(min(xs)), int(max(xs))
    ymin, ymax = int(min(ys)), int(max(ys))
    w, h = pil_img.size
    xmin = max(0, xmin - 3); ymin = max(0, ymin - 3)
    xmax = min(w, xmax + 3); ymax = min(h, ymax + 3)
    return np.array(pil_img.crop((xmin, ymin, xmax, ymax)))


def find_red_box(cv_img, sorted_boxes):
    """Tìm khung vật thể sản phẩm (khung đỏ) bằng Canny + dilate + chấm điểm
    (số khung chữ bên trong, độ ở giữa, độ lớn). Giữ nguyên ý tưởng pipeline."""
    img_h, img_w = cv_img.shape[:2]
    img_area = img_h * img_w
    img_center = (img_w / 2, img_h / 2)
    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (11, 11), 0)
    edges = cv2.Canny(blur, 30, 100)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    dilated = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    red_box = None
    best_score = -1
    for cnt in contours:
        x, y, w, h = cv2.boundingRect(cnt)
        area = w * h
        if area < img_area * 0.05 or area > img_area * 0.65:
            continue
        aspect_ratio = w / float(h)
        if aspect_ratio < 0.15 or aspect_ratio > 3.0:
            continue
        obj_center = (x + w / 2, y + h / 2)
        dist = ((obj_center[0] - img_center[0]) ** 2 + (obj_center[1] - img_center[1]) ** 2) ** 0.5
        max_dist = (img_center[0] ** 2 + img_center[1] ** 2) ** 0.5
        centricity = 1.0 - (dist / max_dist)
        text_inside = 0
        for box in sorted_boxes:
            bx = sum(p[0] for p in box) / 4
            by = sum(p[1] for p in box) / 4
            if x <= bx <= x + w and y <= by <= y + h:
                text_inside += 1
        score = (text_inside * 2.0) + (centricity * 1.5) + (area / img_area)
        if score > best_score:
            best_score = score
            red_box = [x, y, x + w, y + h]
    return red_box


def recognize_vietocr(crop_np):
    """Đọc 1 crop bằng VietOCR (predictor khởi tạo ở cell 7)."""
    try:
        return vietocr_predictor.predict(Image.fromarray(crop_np))
    except Exception:
        return ''


# ---------- Ngưỡng tự tin động + lọc rác (tổng quát) ----------
def _is_valid_pipeline_label(text, field='brand'):
    """Chặn từ khóa rác/quảng cáo/negative — không cho gán vào brand/product mới."""
    if not text or str(text).strip() in ('', ' '):
        return False
    t = unicodedata.normalize('NFC', str(text).strip())
    t_low = t.lower()
    if is_negative_line(t):
        return False
    if any(k in t_low for k in PROMO_KEYWORDS):
        return False
    if re.search(PROMO_PATTERN, t_low):
        return False
    # Bổ sung: % giảm / off (word boundary Unicode hay fail với tiếng Việt)
    if re.search(r'(?:gi[\u1ea3a]m|off|sale|discount)\s*\d+\s*%', t_low):
        return False
    if re.search(r'\d+\s*%\s*(?:off|gi[\u1ea3a]m|sale)?', t_low):
        return False
    if _is_ocr_noise_only(t):
        return False
  # token mạng xã hội đơn lẻ
    words = re.findall(r'[^\W\d_]+', t_low, flags=re.UNICODE)
    if words and all(w in SOCIAL_NOISE_WORDS for w in words):
        return False
    if field == 'brand' and _is_generic_product(t) and len(words) <= 1:
        return False
    if field == 'brand' and not _is_plausible_brand_name(t):
        return False
    if _is_informational_noise(t):
        return False
    if field == 'product' and _is_generic_product(t):
        return False
    if _is_social_caption(t):
        return False
    if _is_description_prose(t):
        return False
    if _is_glued_product_brand(t) and field == 'brand':
        return False
    if field == 'product' and not _is_plausible_product_name(t):
        return False
    return True


def _compute_max_prominence(boxes, texts, img_shape, target_text=None):
    """Tính prominence score cao nhất từ bố cục pipeline (ngưỡng học động / layout bù)."""
    if not boxes or not texts:
        return 0.0
    lines = []
    max_y = 0.0
    for box, text in zip(boxes, texts):
        t = str(text or '').strip()
        if len(t) < 2:
            continue
        ys = [p[1] for p in box]
        height = max(ys) - min(ys)
        y_center = min(ys) + height / 2
        max_y = max(max_y, y_center)
        lines.append({'text': t, 'y_center': y_center, 'height': height})
    if not lines:
        return 0.0
    if max_y <= 0:
        max_y = 1.0
    scores = []
    for ln in lines:
        if is_negative_line(ln['text']):
            continue
        if target_text:
            t_clean = re.sub(r'\W+', '', ln['text'].lower())
            tgt = re.sub(r'\W+', '', target_text.lower())
            if tgt and t_clean != tgt:
                ratio = difflib.SequenceMatcher(None, t_clean, tgt).ratio()
                if ratio < 0.55 and tgt not in t_clean and t_clean not in tgt:
                    continue
        scores.append(compute_brand_prominence_score(ln, max_y))
    return max(scores) if scores else 0.0


def pipeline_predict_with_metrics(pil_img, cv_img, sorted_boxes):
    """ĐỌC TRỰC TIẾP pipeline + trả thêm prominence score (chất lượng bố cục ảnh).
    Trả về (brand_name, product_name, prominence_score)."""
    if not sorted_boxes:
        return '', '', 0.0
    red_box = find_red_box(cv_img, sorted_boxes)
    crops = [crop_box(pil_img, b) for b in sorted_boxes]
    raw_texts = [recognize_vietocr(c) for c in crops]
    cleaned = bilingual_post_processor(raw_texts)
    info = extract_universal_product_info_v5(sorted_boxes, cleaned, cv_img.shape, red_box=red_box)
    brand = str(info.get('brand_name', '') or '').strip()
    product = str(info.get('product_name', '') or '').strip()
    if brand in ('', ' '): brand = ''
    if product in ('', ' '): product = ''
    if brand and not _is_valid_pipeline_label(brand, 'brand'):
        brand = ''
    if product and not _is_valid_pipeline_label(product, 'product'):
        product = ''
    prominence = _compute_max_prominence(sorted_boxes, cleaned, cv_img.shape, brand or None)
    if prominence == 0.0:
        prominence = _compute_max_prominence(sorted_boxes, cleaned, cv_img.shape, None)
    if brand:
        brand = normalize_brand_name(brand)
    if product:
        product = normalize_product_name(product)
        product = _strip_brand_prefix(brand, product)
        if _is_generic_product(product):
            product = ''
    return brand, product, float(prominence)


def pipeline_predict_labels(pil_img, cv_img, sorted_boxes):
    """ĐỌC TRỰC TIẾP theo pipeline (wrapper không cần prominence)."""
    b, p, _ = pipeline_predict_with_metrics(pil_img, cv_img, sorted_boxes)
    return b, p


# Gộp các thương hiệu trong BRAND_NORMALIZATION_MAP của pipeline vào registry
for _c in set(BRAND_NORMALIZATION_MAP.values()):
    _register_brand(_c)

print(f'Lõi pipeline sẵn sàng | registry hiện có {len(BRAND_REGISTRY)} alias thương hiệu.')
print("✓ Đã chạy xong cell 4")

In [ ]:
# ============================================================
# CELL 5 — HUẤN LUYỆN MÔ HÌNH + HỌC ĐỘNG (prominence > DYNAMIC_PROMINENCE_THRESHOLD)
# Học từ train_labels.csv: BrandClassifier + ProductClassifier + BrandKNN.
# Thu thập brand/product MỚI qua pipeline (KHÔNG học ML đơn thuần):
# chỉ đăng ký khi prominence > DYNAMIC_PROMINENCE_THRESHOLD + lọc rác.
# ============================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity


def _build_training_frame():
    """Chuẩn hoá train -> DataFrame với cột _ocr / _brand / _product (auto schema)."""
    if train_labels_df is None:
        return None
    df = train_labels_df.copy()
    oc = OCR_COL if (OCR_COL and OCR_COL in df.columns) else ('ocr_text' if 'ocr_text' in df.columns else None)
    df['_ocr'] = df[oc].astype(str).str.strip() if oc else ''
    if HAS_BRAND_COL:
        df['_brand'] = df[BRAND_COL].astype(str).str.strip().map(normalize_brand_name)
        if PRODUCT_COL is not None:
            df['_product'] = df[PRODUCT_COL].astype(str).str.strip().map(normalize_product_name)
        else:
            df['_product'] = ''
    else:
        merged = df[PRODUCT_COL].astype(str).str.strip() if PRODUCT_COL is not None else pd.Series([''] * len(df))
        bp = merged.map(split_brand_product)
        df['_brand'] = bp.map(lambda t: normalize_brand_name(t[0]))
        df['_product'] = bp.map(lambda t: normalize_product_name(t[1]))
    return df[df['_ocr'] != '']


def _pipe(mf=6000, C=1.5):
    return Pipeline([
        ('tf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=mf, sublinear_tf=True)),
        ('clf', LogisticRegression(max_iter=800, class_weight='balanced', C=C, solver='lbfgs')),
    ])


class BrandClassifier:
    """Brand nặng 0.4 -> ưu tiên evidence (alias trong OCR), classifier là fallback bảo thủ."""
    def __init__(self, min_cls=MIN_BRAND_SUPPORT, thr=0.62):
        self.min_cls = min_cls; self.thr = thr; self._clf = None; self.fitted = False
    def fit(self, df):
        pos = df[df['_brand'] != '']
        vc = pos['_brand'].value_counts()
        keep = vc[vc >= self.min_cls].index
        pos = pos[pos['_brand'].isin(keep)]
        if pos['_brand'].nunique() >= 2:
            self._clf = _pipe(mf=5000).fit(pos['_ocr'], pos['_brand'])
            self.fitted = True
        print('  BrandClassifier: %d dòng, %d brand (support>=%d).' % (len(pos), len(keep), self.min_cls))
        return self
    def predict(self, ocr_text):
        ocr_text = str(ocr_text or '').strip()
        if not ocr_text:
            return ''
        b = detect_brand_in_ocr(ocr_text)
        if b:
            return b
        if self.fitted and self._clf is not None:
            try:
                proba = self._clf.predict_proba([ocr_text])[0]
                i = int(proba.argmax())
                if proba[i] >= self.thr:
                    return normalize_brand_name(str(self._clf.classes_[i]))
            except Exception:
                pass
        return ''


class ProductClassifier:
    """Product nặng 0.25 -> cho phép đoán khi đủ tin cậy, có chặn token generic."""
    def __init__(self, min_cls=2, thr=0.45, gate_thr=0.40):
        self.min_cls = min_cls; self.thr = thr; self.gate_thr = gate_thr
        self._gate = None; self._clf = None; self.fitted = False
    def fit(self, df):
        yg = (df['_product'] != '').astype(int)
        if yg.nunique() >= 2:
            self._gate = _pipe(mf=6000).fit(df['_ocr'], yg)
        pos = df[df['_product'] != '']
        vc = pos['_product'].value_counts()
        pos = pos[pos['_product'].isin(vc[vc >= self.min_cls].index)]
        if pos['_product'].nunique() >= 2:
            self._clf = _pipe(mf=6000).fit(pos['_ocr'], pos['_product'])
            self.fitted = True
        print('  ProductClassifier: %d positive, %d product (support>=%d).' % (
            int(yg.sum()), pos['_product'].nunique(), self.min_cls))
        return self
    def predict(self, ocr_text):
        ocr_text = str(ocr_text or '').strip()
        if not ocr_text or not self.fitted:
            return ''
        try:
            if self._gate is not None:
                gp = self._gate.predict_proba([ocr_text])[0]
                cl = list(self._gate.classes_)
                if 1 in cl and gp[cl.index(1)] < self.gate_thr:
                    return ''
            proba = self._clf.predict_proba([ocr_text])[0]
            i = int(proba.argmax())
            if proba[i] < self.thr:
                return ''
            p = normalize_product_name(str(self._clf.classes_[i]))
            return '' if _is_generic_product(p) else p
        except Exception:
            return ''


class BrandKNN:
    """Fallback similarity: bỏ phiếu brand & product từ các mẫu train gần nhất."""
    def __init__(self, k=5, thr=0.55):
        self.k = k; self.thr = thr; self._tf = None; self._mat = None
        self._brands = []; self._products = []; self.fitted = False
    def fit(self, df):
        self._tf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), max_features=6000, sublinear_tf=True)
        self._mat = self._tf.fit_transform(df['_ocr'])
        self._brands = df['_brand'].tolist()
        self._products = df['_product'].tolist()
        self.fitted = True
        print('  BrandKNN: %d vector.' % len(self._brands))
        return self
    def _vote(self, labels, idx, sims):
        votes = [labels[i] for i in idx if sims[i] >= self.thr and labels[i]]
        return Counter(votes).most_common(1)[0][0] if votes else ''
    def predict(self, ocr_text):
        if not self.fitted or not ocr_text or len(str(ocr_text).strip()) < 5:
            return '', ''
        try:
            sv = self._tf.transform([ocr_text])
            sims = cosine_similarity(sv, self._mat).flatten()
            idx = sims.argsort()[::-1][:self.k]
            return self._vote(self._brands, idx, sims), self._vote(self._products, idx, sims)
        except Exception:
            return '', ''


# ---- Huấn luyện cơ bản từ nhãn CSV ----
brand_clf = product_clf = brand_knn = None
_train_frame = _build_training_frame()

# Đếm support brand/product trong train (phục vụ brand hiếm/mới -> ưu tiên pipeline)
_BRAND_SUPPORT_COUNTS = Counter()
_PRODUCT_SUPPORT_COUNTS = Counter()
if _train_frame is not None and len(_train_frame):
    for b in _train_frame['_brand']:
        if b:
            _BRAND_SUPPORT_COUNTS[_fold_ascii(normalize_brand_name(b))] += 1
    for p in _train_frame['_product']:
        if p:
            _PRODUCT_SUPPORT_COUNTS[_fold_ascii(normalize_product_name(p))] += 1


def _brand_support_count(brand):
    if not brand:
        return 0
    return _BRAND_SUPPORT_COUNTS.get(_fold_ascii(normalize_brand_name(brand)), 0)


def _product_support_count(product):
    if not product:
        return 0
    return _PRODUCT_SUPPORT_COUNTS.get(_fold_ascii(normalize_product_name(product)), 0)


def _is_rare_or_new_brand(brand):
    """Brand xuất hiện ít hoặc chưa từng thấy trong train (< MIN_BRAND_SUPPORT)."""
    if not brand:
        return False
    b = normalize_brand_name(brand)
    if not b:
        return False
    f = _fold_ascii(b)
    if f in BRAND_REGISTRY or f.replace(' ', '') in BRAND_REGISTRY:
        if _brand_support_count(b) >= MIN_BRAND_SUPPORT:
            return False
    return _brand_support_count(b) < MIN_BRAND_SUPPORT


def _is_rare_or_new_product(product):
    if not product:
        return False
    p = normalize_product_name(product)
    return _product_support_count(p) < MIN_BRAND_SUPPORT


def _register_discovered_product(name):
    """Ghi product mới vào bản đồ chuẩn hoá (sau khi qua ngưỡng prominence + lọc rác)."""
    global _PRODUCT_CANON_MAP
    name = unicodedata.normalize('NFC', str(name).strip())
    if not name or not _is_valid_pipeline_label(name, 'product'):
        return False
    _build_product_canon_map()
    key = _fold_ascii(name)
    if key not in _PRODUCT_CANON_MAP:
        _PRODUCT_CANON_MAP[key] = name
        return True
    return False

if _train_frame is not None and len(_train_frame):
    _added = build_brand_knowledge_from_train()
    print('Học từ nhãn CSV: +%d brand từ train (registry=%d alias).' % (_added, len(BRAND_REGISTRY)))
    print('Đang huấn luyện BrandClassifier...')
    brand_clf = BrandClassifier().fit(_train_frame)
    print('Đang huấn luyện ProductClassifier...')
    product_clf = ProductClassifier().fit(_train_frame)
    print('Đang huấn luyện BrandKNN...')
    brand_knn = BrandKNN().fit(_train_frame)
    print('Bộ dự đoán đã sẵn sàng.')
else:
    print('Không có train data -> tắt classifier (chỉ dùng rules).')


def retrain_predictors():
    """Huấn luyện lại 3 mô hình sau khi registry được mở rộng (học động)."""
    global brand_clf, product_clf, brand_knn, _train_frame
    _train_frame = _build_training_frame()
    if _train_frame is None or not len(_train_frame):
        return
    brand_clf = BrandClassifier().fit(_train_frame)
    product_clf = ProductClassifier().fit(_train_frame)
    brand_knn = BrandKNN().fit(_train_frame)


def discover_brands_from_train_images(max_n=None):
    """HỌC ĐỘNG (KHÔNG phải học ML đơn thuần): chạy pipeline trên ảnh train,
    chỉ đăng ký brand/product MỚI khi:
      - prominence score pipeline > DYNAMIC_PROMINENCE_THRESHOLD
      - có bằng chứng trong OCR
      - KHÔNG phải từ khóa rác/quảng cáo (negative filtering)
      - lặp >= MIN_BRAND_SUPPORT lần
  Hàm này cần engine OCR (cell 7) nên được gọi ở cell 7."""
    import random
    if max_n is None:
        max_n = TRAIN_DISCOVERY_SAMPLE
    if SKIP_TRAIN_DISCOVERY or TRAIN_IMAGES_DIR is None:
        print('  Bỏ qua học động (đã tắt hoặc không có thư mục ảnh train).')
        return 0
    paths = [p for p in Path(TRAIN_IMAGES_DIR).glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
    if not paths:
        print('  Không tìm thấy ảnh train -> bỏ qua học động.')
        return 0
    random.seed(42); random.shuffle(paths)
    paths = paths[:max_n]
    brand_cand = Counter()
    product_cand = Counter()
    n_high_prom = 0
    _build_product_canon_map()
    for p in tqdm(paths, desc='Học động (ảnh train)'):
        try:
            pil = preprocess(Image.open(str(p)).convert('RGB'))
            cv_img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
            boxes = _pipeline_sorted_boxes(cv_img, ocr_det)
            if not boxes:
                continue
            txt, conf = _vietocr_pipeline_ocr(pil, boxes)
            if not str(txt).strip():
                continue
            b, pr, prominence = pipeline_predict_with_metrics(pil, cv_img, boxes)
            if prominence < DYNAMIC_PROMINENCE_THRESHOLD:
                continue
            n_high_prom += 1
            folded_txt = _fold_ascii(txt)
            if b and _is_valid_pipeline_label(b, 'brand'):
                bf = _fold_ascii(b)
                known = bf in BRAND_REGISTRY or bf.replace(' ', '') in BRAND_REGISTRY
                if not known and len(bf.replace(' ', '')) >= MIN_BRAND_ALIAS_LEN:
                    if bf.split()[0] in folded_txt:
                        brand_cand[b] += 1
            if pr and _is_valid_pipeline_label(pr, 'product'):
                pk = _fold_ascii(normalize_product_name(pr))
                if pk not in _PRODUCT_CANON_MAP and _fold_ascii(pr.split()[0]) in folded_txt:
                    product_cand[pr] += 1
        except Exception:
            continue
    added_b, added_p = 0, 0
    for b, c in brand_cand.items():
        if c >= MIN_BRAND_SUPPORT:
            _register_brand(b)
            added_b += 1
    for pr, c in product_cand.items():
        if c >= MIN_BRAND_SUPPORT and _register_discovered_product(pr):
            added_p += 1
    print(f'  Học động: ảnh prominence>{DYNAMIC_PROMINENCE_THRESHOLD}: {n_high_prom}')
    print(f'  Đăng ký mới: +{added_b} brand, +{added_p} product (ứng viên brand={len(brand_cand)}, product={len(product_cand)}).')
    return added_b + added_p


print("✓ Đã chạy xong cell 5")

In [ ]:
# ============================================================
# CELL 6 — DỰ ĐOÁN (have_train-first) + EVAL + KIỂM THỬ
# predict_labels: rules -> classifier -> KNN (giống have_train).
# predict_image: ML là nguồn chính; pipeline layout chỉ bù khi prominence cao.
# ============================================================

def _strip_brand_prefix(brand, product):
    """Bỏ token brand bị lặp ở đầu product (vd brand 'Vinamilk', product 'Vinamilk Dielac')."""
    if not (brand and product):
        return product
    bf = _fold_ascii(brand)
    pf = _fold_ascii(product)
    if pf == bf:
        return ''
    if pf.startswith(bf + ' '):
        return product[len(brand):].strip()
    return product


def predict_labels(ocr_text, image_id=None):
    """Dự đoán brand/product từ văn bản OCR (rules -> classifier -> KNN)."""
    ocr_text = str(ocr_text or '').strip()
    if _is_ocr_noise_only(ocr_text) or _is_informational_noise(ocr_text):
        return '', ''

    brand = product = ''

    merged = extract_by_rules(ocr_text)
    if merged:
        merged = post_process_prediction(merged, ocr_text)
        brand, product = split_brand_product(merged)

    if not brand:
        brand = detect_brand_in_ocr(ocr_text)

    if not brand and brand_clf is not None:
        brand = brand_clf.predict(ocr_text)

    if not product:
        product = guess_product_from_ocr(ocr_text, brand)

    if not product and product_clf is not None:
        product = product_clf.predict(ocr_text)

    if (not brand or not product) and brand_knn is not None:
        kb, kp = brand_knn.predict(ocr_text)
        if not brand and kb:
            brand = normalize_brand_name(kb)
        if not product and kp and not _is_generic_product(kp):
            product = normalize_product_name(kp)

    brand, product = reconcile_brand_product(ocr_text, brand, product)
    return brand, product


def _normalize_pair(brand, product):
    brand = normalize_brand_name(brand) if brand else ''
    if product:
        product = normalize_product_name(product)
        product = _strip_brand_prefix(brand, product)
        if _is_generic_product(product) or _is_description_prose(product):
            product = ''
    if _is_glued_product_brand(brand) or _is_description_prose(brand):
        brand = ''
    return brand, product


def predict_image(image_id, ocr_text, ocr_conf, boxes=None):
    """have_train-first: predict_labels trên OCR VietOCR là nguồn chính.
    Pipeline layout chỉ bù khi prominence >= PIPELINE_LAYOUT_MIN_PROMINENCE
    và ML trống hoặc brand/product hiếm/chưa có đủ support trong train."""
    cleaned = clean_social_ocr(ocr_text)
    if _is_ocr_noise_only(cleaned) or _is_informational_noise(cleaned):
        return '', '', False

    brand, product = predict_labels(cleaned, image_id)

    pipe_brand, pipe_product, prominence = '', '', 0.0
    used_pipeline = False

    if boxes:
        try:
            pil_img = load_image(image_id)
            if pil_img is not None:
                pil_img = preprocess(pil_img)
                cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
                pipe_brand, pipe_product, prominence = pipeline_predict_with_metrics(
                    pil_img, cv_img, boxes)
                used_pipeline = True
        except Exception:
            pipe_brand, pipe_product, prominence = '', '', 0.0

    if used_pipeline and prominence >= PIPELINE_LAYOUT_MIN_PROMINENCE:
        pb_ok = pipe_brand and _is_valid_pipeline_label(pipe_brand, 'brand')
        pp_ok = pipe_product and _is_valid_pipeline_label(pipe_product, 'product')

        if pb_ok and not _is_glued_product_brand(pipe_brand) and not _is_description_prose(pipe_brand):
            if not brand or _is_rare_or_new_brand(brand):
                brand = pipe_brand
        if pp_ok and _is_plausible_product_name(pipe_product):
            if (not product or _is_social_caption(product)
                    or _is_rare_or_new_product(product) or _is_generic_product(product)):
                product = pipe_product

    brand, product = reconcile_brand_product(cleaned, brand, product)
    brand, product = _normalize_pair(brand, product)
    return brand, product, used_pipeline


def _token_f1(pred, gt):
    if not gt and not pred:
        return 1.0
    if not gt or not pred:
        return 0.0
    pt = _fold_ascii(pred).split()
    gt_ = _fold_ascii(gt).split()
    tp = sum(1 for t in pt if t in gt_)
    if tp == 0:
        return 0.0
    p = tp / len(pt)
    r = tp / len(gt_)
    return 2 * p * r / (p + r)


def evaluate_pipeline(df=None, n=500, assumed_cer=0.25):
    if df is None:
        df = _train_frame
    if df is None or not len(df):
        return None
    if len(df) > n:
        df = df.sample(n, random_state=42)
    sb = sp = exact_b = exact_p = ev = 0
    for _, row in df.iterrows():
        ocr = str(row['_ocr']).strip()
        gt_b = str(row['_brand']).strip()
        gt_p = str(row['_product']).strip()
        pb, pp = predict_labels(ocr)
        sb += _token_f1(pb, gt_b)
        sp += _token_f1(pp, gt_p)
        if _fold_ascii(pb) == _fold_ascii(gt_b):
            exact_b += 1
        if _fold_ascii(pp) == _fold_ascii(gt_p):
            exact_p += 1
        ev += 1
    f1b = sb / ev if ev else 0
    f1p = sp / ev if ev else 0
    return {
        'F1_brand': f1b, 'F1_product': f1p,
        'ExactBrand': exact_b / ev if ev else 0,
        'ExactProduct': exact_p / ev if ev else 0,
        'N': ev, 'AssumedCER': assumed_cer,
    }


if _train_frame is not None and len(_train_frame):
    print('Eval %d mẫu train (predict_labels trên OCR train)...' % min(500, len(_train_frame)))
    m = evaluate_pipeline(_train_frame, 500)
    print('F1_brand=%.4f  F1_product=%.4f  ExactBrand=%.4f  ExactProduct=%.4f  N=%d' % (
        m['F1_brand'], m['F1_product'], m['ExactBrand'], m['ExactProduct'], m['N']))
    for _cer in (0.0, 0.15, 0.25):
        _s = 0.4 * m['F1_brand'] + 0.35 * (1 - _cer) + 0.25 * m['F1_product']
        print('  Est.Score (CER=%.2f): %.4f' % (_cer, _s))

_register_brand('Dove')
_P = [
    ('HALONG CANFOCO Pate C\u1ed9t \u0110\u00e8n', 'Ha Long Canfoco', 'Pate'),
    ('ate Jate Cotcen 130 tan thit lon',           None,             'Pate C\u1ed9t \u0110\u00e8n'),
    ('HIGHLANDS COFFEE tra sen vang',              'Highlands Coffee', None),
    ('tiktok viral capcut fyp news',               '',               ''),
    ('HiPP Combiotic organic milk',                'HiPP',           None),
    ('Nestl\u00e9 NAN OPTIpro 0-6',                'Nestl\u00e9',    'NAN'),
    ('Dove Smoothie t\u1ea9y da ch\u1ebft',        'Dove',           None),
    ('Vinamilk Dielac so 1',                        'Vinamilk',       'Dielac'),
    ('Trả lời hình luận của Thúy nêng chốt liền rẻ quá Abbott Ped Complete Ion Pediasure Úc 850g',
                                                   'Abbott',         'PediaSure'),
    ('Similac Totalladongsua cong thuc cao cap cua Abbott, duoc phat trien nhâm ho tro',
                                                   'Abbott',         'Similac'),
    ('1964 Không chi là Quốc Tế Thiếu Nhi Vitamin A EM SỮA 14 THÁNG các mom đừng quên ngày mai',
                                                   '',               ''),
]
print('Kiểm thử nhanh predict_labels:')
ok_n = 0
for txt, eb, ec in _P:
    b, p = predict_labels(txt)
    combined = (b + ' ' + p).strip()
    ok = True
    if eb is not None:
        ok = ok and ((eb == '' and b == '') or (eb != '' and eb.lower() in b.lower()))
    if ec is not None:
        ok = ok and ((ec == '' and combined == '') or (ec != '' and ec.lower() in combined.lower()))
    if p and _is_generic_product(p):
        ok = False
    if ok:
        ok_n += 1
    print('  [%s] %-40s => brand=%-16s product=%s' % (
        'OK' if ok else 'FAIL', txt[:40], repr(b)[:16], repr(p)[:24]))
print('  Kết quả: %d/%d' % (ok_n, len(_P)))
print("✓ Đã chạy xong cell 6")

In [ ]:
# ============================================================
# CELL 7 — ENGINE OCR: Paddle det ONLY + VietOCR (không latin rec)
# Bound box: PP-OCRv4 det. Đọc chữ: VietOCR từng crop.
# ============================================================
import os
from pathlib import Path
import re
import numpy as np
import cv2
from PIL import Image, ImageEnhance, ImageFilter
import torch
from concurrent.futures import ThreadPoolExecutor, as_completed

def _patch_numpy_for_imgaug():
    """PaddleOCR 2.7.3 -> imgaug dùng np.sctypes (đã bỏ ở numpy 2.x)."""
    import numpy as _np
    if not hasattr(_np, 'sctypes'):
        _np.sctypes = {
            'float': [_np.float16, _np.float32, _np.float64],
            'int': [_np.int8, _np.int16, _np.int32, _np.int64],
            'uint': [_np.uint8, _np.uint16, _np.uint32, _np.uint64],
            'complex': [_np.complex64, _np.complex128],
            'others': [bool, _np.bytes_, _np.str_, _np.object_],
        }

_patch_numpy_for_imgaug()

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

_pp_det = None


def _det_model_dir():
    return Path.home() / '.paddleocr' / 'whl' / 'det' / 'ch' / 'ch_PP-OCRv4_det_infer'


def _paddle_model_ready(d):
    d = Path(d)
    return (d / 'inference.pdmodel').is_file() and (d / 'inference.pdiparams').is_file()


def _get_maybe_download():
    """ppocr chỉ có sau khi paddleocr được import — thử nhiều đường dẫn."""
    import importlib
    import paddleocr  # noqa: F401 — đưa ppocr vào sys.path trên một số bản cài
    for mod_name in ('paddleocr.ppocr.utils.network', 'ppocr.utils.network'):
        try:
            return importlib.import_module(mod_name).maybe_download
        except ModuleNotFoundError:
            continue
    raise RuntimeError(
        'Không tìm thấy ppocr.utils.network. Chạy lại cell 1 (paddleocr==2.7.3) rồi Restart session.'
    )


def _prefetch_det_model():
    det_dir = _det_model_dir()
    url = 'https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_det_infer.tar'
    if not _paddle_model_ready(det_dir):
        print('Tải trước model det PP-OCRv4...')
        _get_maybe_download()(str(det_dir), url)
        print('  đã tải det')
    return det_dir


def _make_paddle_det():
    """Det PP-OCRv4 — chỉ text_detector, không rec latin."""
    _patch_numpy_for_imgaug()
    import paddleocr  # phải import trước prefetch / ppocr
    det_dir = None
    try:
        det_dir = _prefetch_det_model()
    except Exception as e:
        print(f'  Prefetch det bỏ qua ({e}) — PaddleOCR sẽ tự tải model khi khởi tạo.')
    
    from paddleocr import PaddleOCR
    # Cấu hình rõ ràng chỉ sử dụng Detector, bỏ qua Rec và Cls để tránh tải thêm file thừa
    return PaddleOCR(
        ocr_version='PP-OCRv4',
        det=True,
        rec=False,         # Tắt bộ nhận diện của Paddle
        cls=False,         # Tắt bộ phân loại góc của Paddle
        use_angle_cls=False,
        det_model_dir=str(det_dir) if det_dir else None,  # Chỉ định thư mục chứa det model đã tải
        lang='ch',         # Đặt 'ch' khớp với thư mục mô hình det tiếng Trung/Đa ngôn ngữ của PP-OCRv4
        use_gpu=False,
        show_log=False,
        enable_mkldnn=True,
    )


def preprocess(img, max_dim=1280):
    w, h = img.size
    if max(w, h) > max_dim:
        r = max_dim / max(w, h)
        img = img.resize((int(w * r), int(h * r)), Image.LANCZOS)
    img = ImageEnhance.Contrast(img).enhance(1.35)
    return img.filter(ImageFilter.SHARPEN)


def postprocess_ocr(text):
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    if not tokens:
        return ''
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower():
            deduped.append(tok)
    return ' '.join(deduped)


def _pipeline_sorted_boxes(cv_img, det_engine):
    """Bound khung chữ: text_detector + lọc theo OCR_BOX_MIN_AREA_RATIO."""
    try:
        dt_boxes, _ = det_engine.text_detector(cv_img)
        raw_boxes = dt_boxes.tolist() if dt_boxes is not None else []
    except Exception:
        return []
    img_h, img_w = cv_img.shape[:2]
    img_area = img_h * img_w
    min_ratio = OCR_BOX_MIN_AREA_RATIO
    valid_boxes = [
        box for box in raw_boxes
        if ((max(p[0] for p in box) - min(p[0] for p in box))
            * (max(p[1] for p in box) - min(p[1] for p in box)) / img_area) >= min_ratio
    ]
    return sorted(valid_boxes, key=lambda box: (box[0][1], box[0][0]))


def _vietocr_pipeline_ocr(pil_img, boxes):
    """VietOCR từng crop + bilingual_post_processor — nguồn OCR chính cho predict_labels."""
    if not boxes:
        return '', 0.0
    raw_texts = []
    for box in boxes:
        crop_np = crop_box(pil_img, box)
        if crop_np.size == 0:
            raw_texts.append('')
            continue
        raw_texts.append(recognize_vietocr(crop_np))
    cleaned = bilingual_post_processor(raw_texts)
    ocr_text = postprocess_ocr(' '.join(t for t in cleaned if str(t).strip()))
    return ocr_text, (0.85 if ocr_text.strip() else 0.0)


def _process_one_image(image_id):
    """Det (Paddle) + VietOCR trên luồng chính."""
    path = IMAGE_PATH_INDEX.get(str(image_id))
    if not path or not os.path.isfile(path):
        return image_id, '', 0.0, []
    try:
        pil = preprocess(Image.open(path).convert('RGB'))
    except Exception:
        return image_id, '', 0.0, []
    cv_img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    boxes = _pipeline_sorted_boxes(cv_img, ocr_det)
    ocr_text, conf = _vietocr_pipeline_ocr(pil, boxes)
    return image_id, ocr_text, conf, boxes


def _init_ocr_process():
    global _pp_det
    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['MKL_NUM_THREADS'] = '1'
    _patch_numpy_for_imgaug()
    _pp_det = _make_paddle_det()


def _det_image_process(image_id, image_path):
    global _pp_det
    if not image_path or not os.path.isfile(image_path):
        return image_id, []
    try:
        pil = preprocess(Image.open(image_path).convert('RGB'))
    except Exception:
        return image_id, []
    cv_img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    boxes = _pipeline_sorted_boxes(cv_img, _pp_det)
    return image_id, boxes


def load_image(image_id):
    path = IMAGE_PATH_INDEX.get(str(image_id))
    if path is None or not Path(path).exists():
        return None
    try:
        return Image.open(str(path)).convert('RGB')
    except Exception:
        return None


print('Đang nạp Paddle det PP-OCRv4 (chỉ bound box, không latin rec)...')
ocr_det = _make_paddle_det()
print('Det sẵn sàng | OCR text: VietOCR trên crop.')

print('Đang nạp VietOCR (vgg_seq2seq, CPU)...')
torch.set_num_threads(max(1, CPU_THREADS // 2))
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
_vcfg = Cfg.load_config_from_name('vgg_seq2seq')
_vcfg['device'] = 'cpu'
vietocr_predictor = Predictor(_vcfg)
print('VietOCR sẵn sàng.')

print('\nChạy học động (thu thập brand mới từ ảnh train)...')
_n_new = discover_brands_from_train_images()
if _n_new:
    print('Huấn luyện lại mô hình sau học động...')
    retrain_predictors()

print('\nKiểm thử nhanh trên ảnh test đầu tiên:')
_fid = test_df['image_id'].iloc[0]
_img = load_image(_fid)
if _img is not None:
    _pil = preprocess(_img)
    _cv = cv2.cvtColor(np.array(_pil), cv2.COLOR_RGB2BGR)
    _boxes = _pipeline_sorted_boxes(_cv, ocr_det)
    _txt, _conf = _vietocr_pipeline_ocr(_pil, _boxes)
    _b, _p = predict_labels(clean_social_ocr(_txt))
    print(f'  image_id : {_fid}')
    print(f'  khung    : {len(_boxes)} box (Paddle det)')
    print(f'  ocr_text : {_txt[:80]}{"..." if len(_txt) > 80 else ""}')
    print(f'  predict  : brand={repr(_b)[:20]} product={repr(_p)[:30]}')
else:
    print(f'  Cảnh báo: không tìm thấy ảnh {_fid}')
print("✓ Đã chạy xong cell 7")

In [ ]:
# ============================================================
# CELL 8 — CHẠY OCR THEO LÔ + DỰ ĐOÁN LAI + LƯU CHECKPOINT
# Det + VietOCR trên luồng chính (ocr_det cell 7). Loky worker det lỗi im lặng trên Kaggle.
# ============================================================

SAVE_EVERY = 50
RESET_CHECKPOINT = False   # True = xóa checkpoint cũ, chạy lại từ đầu; xong thì đổi False

if RESET_CHECKPOINT and CHECKPOINT_CSV.exists():
    CHECKPOINT_CSV.unlink()
    print(f'Đã xóa checkpoint cũ: {CHECKPOINT_CSV}')

# Khôi phục từ checkpoint nếu có
results, done_ids = [], set()
if CHECKPOINT_CSV.exists():
    ckpt = pd.read_csv(CHECKPOINT_CSV, keep_default_na=False)
    done_ids = set(ckpt['image_id'])
    results = ckpt.to_dict('records')
    print(f'Tiếp tục từ checkpoint: {len(done_ids):,} ảnh đã xong.')
else:
    print('Bắt đầu chạy mới.')

pending = [i for i in test_df['image_id'] if i not in done_ids]
print(f'Còn lại: {len(pending):,} ảnh | det+VietOCR tuần tự (luồng chính)')

errors = 0
n_pipeline = 0
n_viet_read = 0
t0 = time.perf_counter()

for batch_start in tqdm(range(0, len(pending), SAVE_EVERY), desc='OCR theo lô'):
    batch = pending[batch_start: batch_start + SAVE_EVERY]

    for image_id in batch:
        try:
            image_id, ocr_text, conf, boxes = _process_one_image(image_id)
            if ocr_text.strip():
                n_viet_read += 1
            brand, product, used = predict_image(image_id, ocr_text, conf, boxes)
            if used:
                n_pipeline += 1
            results.append({
                'image_id': image_id,
                'ocr_text': ocr_text,
                'brand_name': brand,
                'product_name': product,
            })
        except Exception:
            results.append({'image_id': image_id, 'ocr_text': '',
                            'brand_name': '', 'product_name': ''})
            errors += 1

    pd.DataFrame(results).to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8')

elapsed = time.perf_counter() - t0
n_ocr   = sum(1 for r in results if str(r.get('ocr_text', '')).strip())
n_brand = sum(1 for r in results if str(r.get('brand_name', '')).strip())
n_prod  = sum(1 for r in results if str(r.get('product_name', '')).strip())
print(f'\nHoàn tất {len(results)} ảnh trong {elapsed/60:.1f} phút ({elapsed/max(len(results),1):.2f}s/ảnh)')
print(f'OCR có chữ: {n_ocr} | Có brand: {n_brand} | Có product: {n_prod}')
print(f'VietOCR có chữ: {n_viet_read} ảnh | Có box det (layout bù): {n_pipeline} | Lỗi: {errors}')
print("✓ Đã chạy xong cell 8")

In [ ]:
# ============================================================
# CELL 9 — HẬU KỲ NHẸ (trước submission)
# Chỉ bỏ token lặp liền kề — không cắt max 5 token (giữ tên product đầy đủ).
# ============================================================

def _sanitize_product_name(product):
    product = str(product or '').strip()
    if not product:
        return ''
    tokens = product.split()
    deduped = [tokens[0]]
    for tok in tokens[1:]:
        if tok.lower() != deduped[-1].lower():
            deduped.append(tok)
    return ' '.join(deduped)


def postprocess_submission_df(df):
    df = df.copy()
    if 'product_name' in df.columns:
        before = df['product_name'].astype(str)
        df['product_name'] = before.map(_sanitize_product_name)
    return df


results = globals().get('results')
_ckpt = globals().get('CHECKPOINT_CSV', Path('/kaggle/working/checkpoint.csv'))
if not results and Path(_ckpt).exists():
    results = pd.read_csv(_ckpt, keep_default_na=False).to_dict('records')
    print(f'Đã nạp {len(results):,} dòng từ checkpoint.')
if not results:
    results = []

_df = pd.DataFrame(results)
if _df.empty:
    print('Chưa có kết quả — chạy cell 8 trước.')
else:
    _before = _df.get('product_name', pd.Series(dtype=str)).astype(str)
    _df = postprocess_submission_df(_df)
    _changed = int((_before != _df['product_name'].astype(str)).sum())
    results = _df.to_dict('records')
    pd.DataFrame(results).to_csv(_ckpt, index=False, encoding='utf-8')
    print(f'Đã hậu kỳ product_name: {_changed:,} dòng thay đổi / {len(_df):,} ảnh.')
print("✓ Đã chạy xong cell 9")

In [ ]:
# ============================================================
# CELL 10 — XUẤT submission.csv ĐÚNG CHUẨN CUỘC THI (have_train)
# Khớp đúng schema/thứ tự cột của sample_submission, thay rỗng = ' ',
# bỏ ký tự xuống dòng, kiểm 7 ràng buộc (AC1-AC7) trước khi ghi.
# (Phần này GIỮ NGUYÊN chuẩn have_train, không thay đổi.)
# ============================================================
import csv as _csv

sample = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
sample_cols = list(sample.columns)


def _pick(cols, *cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low:
            return low[c.lower()]
    return None

ID_OUT      = _pick(sample_cols, 'image_id', 'id') or 'image_id'
OCR_OUT     = _pick(sample_cols, 'ocr_text', 'ocr', 'text')
BRAND_OUT   = _pick(sample_cols, 'brand_name', 'brand')
PRODUCT_OUT = _pick(sample_cols, 'product_name', 'product')

res_df = pd.DataFrame(results)
sub = pd.DataFrame({ID_OUT: res_df['image_id']})
if OCR_OUT:     sub[OCR_OUT]     = res_df.get('ocr_text', '')
if BRAND_OUT:   sub[BRAND_OUT]   = res_df.get('brand_name', '')
if PRODUCT_OUT: sub[PRODUCT_OUT] = res_df.get('product_name', '')

text_cols = [c for c in (OCR_OUT, BRAND_OUT, PRODUCT_OUT) if c]
for col in text_cols:
    sub[col] = sub[col].fillna('').astype(str).str.replace(r'[\n\t\r]', ' ', regex=True).str.strip()
    sub.loc[sub[col] == '', col] = ' '

for c in sample_cols:
    if c not in sub.columns:
        sub[c] = ' '
sub = sub[sample_cols]

checks = {
    'AC1 đủ số dòng':      len(sub) == len(sample),
    'AC2 không dư id':     not bool(set(sub[ID_OUT]) - set(sample[ID_OUT])),
    'AC3 không thiếu id':  not bool(set(sample[ID_OUT]) - set(sub[ID_OUT])),
    'AC4 không trùng id':  not sub[ID_OUT].duplicated().any(),
    'AC5 không null':      not sub.isnull().any().any(),
    'AC6 không xuống dòng': not any(sub[c].str.contains(r'[\n\t\r]', regex=True, na=False).any() for c in text_cols),
    'AC7 đúng thứ tự cột': list(sub.columns) == sample_cols,
}
ok = all(checks.values())
for k, v in checks.items():
    print('  [%s] %s' % ('ĐẠT' if v else 'LỖI', k))

if ok:
    sub = sub.set_index(ID_OUT).reindex(sample[ID_OUT]).reset_index()
    sub = sub[sample_cols]
    sub.to_csv(OUTPUT_CSV, index=False, encoding='utf-8', quoting=_csv.QUOTE_ALL)
    print('Đã lưu: %s (%.1f KB) | cột=%s' % (OUTPUT_CSV, OUTPUT_CSV.stat().st_size / 1024, sample_cols))
    if BRAND_OUT:
        nb = (sub[BRAND_OUT].str.strip() == '').sum()
        print('Không có brand : %d (%.1f%%)' % (nb, nb / len(sub) * 100))
        top_b = sub[sub[BRAND_OUT].str.strip() != ''][BRAND_OUT].str.strip().value_counts().head(10)
        print('Top brand:')
        for nm, cnt in top_b.items(): print('  %5dx %s' % (cnt, nm))
    if PRODUCT_OUT:
        npd = (sub[PRODUCT_OUT].str.strip() == '').sum()
        print('Không có product : %d (%.1f%%)' % (npd, npd / len(sub) * 100))
        top_p = sub[sub[PRODUCT_OUT].str.strip() != ''][PRODUCT_OUT].str.strip().value_counts().head(10)
        print('Top product:')
        for nm, cnt in top_p.items(): print('  %5dx %s' % (cnt, nm))
else:
    print('[LỖI] Một số ràng buộc chưa đạt - chưa ghi file!')
print("✓ Đã chạy xong cell 10")